# mini-k3: compact-model speed and quality study

This clean notebook reproduces the internally fixed development screen, locked confirmation, additional-domain check and repeated timing validation. It contains exact source and no invented execution outputs. Use a fresh Colab T4 runtime. Inspect `PROTOCOL.md`, `OOD_PROTOCOL.md` and `TIMING_PROTOCOL.md` before running. The experiment trains 15 small models; free GPU access and session lifetime are variable.


## Recorded result from 18 October 2026

The following is a summary of the saved measured evidence, separate from this notebook’s fresh execution.

| Model | Parameters | Mean heldout perplexity | Mean training tokens/s |
| --- | ---: | ---: | ---: |
| baseline6 | 13,227,648 | 9.794 | 119,735 |
| baseline5 | 11,432,704 | 9.913 | 142,115 |
| attnres5_b12 | 11,441,152 | 9.409 | 134,756 |

Read `analysis/PHASE3_REPORT.md` for all seeds, additional-domain failures and timing limitations.


In [ ]:
import sys,torch
print(sys.version,torch.__version__)
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))


In [ ]:
import os,subprocess,urllib.request,hashlib,json
from pathlib import Path
WORK=Path('/content/mini_k3_work')
WORK.mkdir(exist_ok=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.6','datasets==3.6.0','matplotlib'])
os.environ.update(USE_TF='0',USE_FLAX='0',HF_HUB_DISABLE_XET='1',TOKENIZERS_PARALLELISM='false')
os.environ['PYTHONPATH']=str(WORK/'vendor')+os.pathsep+str(WORK)
EXPECTED={'model/model_minimind.py': '548f1203282dccb3be43227b11463f469b976afbab614df52828bdd246c3786f', 'model/tokenizer.json': '71f32c68cf63a15355a8fc171b7594b3d41870fe0ddb54fc6aefa55f73a4a668', 'model/tokenizer_config.json': 'd7cd6a60c9f191c4f9ffec69b2eb289ad2f960366724f328b681ae1e46e4c110', 'LICENSE': 'c71d239df91726fc519c6eb72d318ec65820627232b2f796219e87dcf35d0ab4'}
for name,expected in EXPECTED.items():
    target=WORK/'vendor'/name
    target.parent.mkdir(parents=True,exist_ok=True)
    data=urllib.request.urlopen('https://raw.githubusercontent.com/jingyaogong/minimind/f659b55761b754d306bd140573493a6543cafd7f/'+name).read()
    assert hashlib.sha256(data).hexdigest()==expected
    target.write_bytes(data)
PAYLOAD={'attnres.py': '"""Block Attention Residuals adapted to the pinned MiniMind backbone.\n\nReference: Kimi K3, section 2.2, equations 8--10:\nhttps://arxiv.org/html/2607.24653v2#S2.SS2\nThis small PyTorch implementation is not Moonshot\'s optimized kernel.\n"""\nimport torch\nfrom torch import nn\nfrom model.model_minimind import MiniMindModel, RMSNorm\n\n\nclass DepthMixer(nn.Module):\n    def __init__(self, width, eps=1e-6):\n        super().__init__()\n        self.query = nn.Parameter(torch.zeros(width))\n        self.key_norm = RMSNorm(width, eps)\n\n    def forward(self, sources):\n        # sources: list of [batch, sequence, width]. Softmax is over depth.\n        values = torch.stack(sources, dim=0)\n        keys = self.key_norm(values).float()\n        scores = (keys * self.query.float()).sum(dim=-1)\n        weights = scores.softmax(dim=0)\n        return (values.float() * weights.unsqueeze(-1)).sum(dim=0).to(values.dtype)\n\n\nclass BlockAttnResModel(MiniMindModel):\n    def __init__(self, config, block_sublayers=4):\n        super().__init__(config)\n        if block_sublayers < 1 or config.use_moe:\n            raise ValueError(\'This pilot supports dense models and positive block sizes.\')\n        self.block_sublayers = block_sublayers\n        self.depth_mixers = nn.ModuleList([\n            DepthMixer(config.hidden_size, config.rms_norm_eps)\n            for _ in range(2 * config.num_hidden_layers + 1)\n        ])\n\n    def forward(self, input_ids, attention_mask=None, past_key_values=None,\n                use_cache=False, **kwargs):\n        if use_cache or past_key_values is not None:\n            raise NotImplementedError(\'Pilot backbone has no KV cache; use_cache=False.\')\n        seq = input_ids.shape[1]\n        if seq > self.config.max_position_embeddings:\n            raise ValueError(\'Sequence exceeds configured context length.\')\n        embedding = self.dropout(self.embed_tokens(input_ids))\n        positions = (self.freqs_cos[:seq], self.freqs_sin[:seq])\n        completed, partial, index = [embedding], None, 0\n        for layer in self.layers:\n            for kind in (\'attention\', \'mlp\'):\n                sources = completed + ([] if partial is None else [partial])\n                hidden = self.depth_mixers[index](sources)\n                if kind == \'attention\':\n                    delta, _ = layer.self_attn(\n                        layer.input_layernorm(hidden), positions,\n                        use_cache=False, attention_mask=attention_mask)\n                else:\n                    delta = layer.mlp(layer.post_attention_layernorm(hidden))\n                partial = delta if partial is None else partial + delta\n                index += 1\n                if index % self.block_sublayers == 0:\n                    completed.append(partial)\n                    partial = None\n        sources = completed + ([] if partial is None else [partial])\n        hidden = self.norm(self.depth_mixers[index](sources))\n        return hidden, [None] * len(self.layers), hidden.new_zeros(())\n', 'experiment.py': '"""Resumable matched-budget MiniMind / Block AttnRes pilot. No results are assumed."""\nimport argparse\nimport contextlib\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport statistics\nimport subprocess\nimport time\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom transformers import AutoTokenizer\nfrom model.model_minimind import MiniMindConfig, MiniMindForCausalLM\nfrom attnres import BlockAttnResModel\n\nUPSTREAM = \'f659b55761b754d306bd140573493a6543cafd7f\'\nDATASET = \'roneneldan/TinyStories\'\nDATA_REVISION = \'f54c09fd23315a6f9c86f9dc80f725de7d8f9c64\'\n\n\ndef dump_json(path, value):\n    path = Path(path)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n    os.replace(temp, path)\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef make_config(size, seq):\n    sizes = {\'tiny\': (64, 2, 4, 2), \'pilot\': (384, 6, 6, 2),\n             \'standard\': (768, 8, 8, 4)}\n    width, depth, heads, kv = sizes[size]\n    return MiniMindConfig(hidden_size=width, num_hidden_layers=depth,\n                          num_attention_heads=heads, num_key_value_heads=kv,\n                          max_position_embeddings=max(seq, 512),\n                          dropout=0.0, use_moe=False)\n\n\ndef build_model(size, seq, variant, seed, block_sublayers=4):\n    torch.manual_seed(seed)\n    config = make_config(size, seq)\n    baseline = MiniMindForCausalLM(config)\n    if variant == \'baseline\':\n        return baseline\n    # Copy shared weights exactly; only depth mixing is newly initialized.\n    backbone = BlockAttnResModel(config, block_sublayers)\n    missing, unexpected = backbone.load_state_dict(baseline.model.state_dict(), strict=False)\n    assert not unexpected and all(k.startswith(\'depth_mixers.\') for k in missing)\n    baseline.model = backbone\n    if config.tie_word_embeddings:\n        baseline.model.embed_tokens.weight = baseline.lm_head.weight\n    return baseline\n\n\ndef prepare_data(root, upstream_dir, train_tokens=2_000_000, val_tokens=100_000):\n    """Cache disjoint, exact-text-deduplicated official train/validation prefixes."""\n    from datasets import load_dataset\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    manifest_path = root / \'data_manifest.json\'\n    tokenizer = AutoTokenizer.from_pretrained(str(Path(upstream_dir) / \'model\'),\n                                               trust_remote_code=False)\n    spec = {\'dataset\': DATASET, \'revision\': DATA_REVISION,\n            \'train_token_cap\': train_tokens, \'validation_token_cap\': val_tokens,\n            \'tokenizer_sha256\': digest(Path(upstream_dir) / \'model/tokenizer.json\'),\n            \'selection\': \'official split prefixes; exact-text dedup; EOS between stories\'}\n    if manifest_path.exists():\n        old = json.loads(manifest_path.read_text())\n        if old[\'spec\'] == spec:\n            for split in (\'train\', \'validation\'):\n                if digest(root / f\'{split}.npy\') != old[\'files\'][split][\'sha256\']:\n                    raise RuntimeError(\'Cached token file changed. Use a new data directory.\')\n            print(\'Using verified token cache:\', root, flush=True)\n            return old\n        raise RuntimeError(\'Data configuration changed. Choose a new data directory.\')\n    seen, files = set(), {}\n    for split, cap in [(\'train\', train_tokens), (\'validation\', val_tokens)]:\n        stream = load_dataset(DATASET, revision=DATA_REVISION, split=split, streaming=True)\n        tokens, count = [], 0\n        for row in stream:\n            text = row[\'text\'].strip()\n            if not text:\n                continue\n            h = hashlib.sha256(text.encode(\'utf-8\')).digest()\n            if h in seen:\n                continue\n            seen.add(h)\n            tokens.extend(tokenizer.encode(text, add_special_tokens=False))\n            tokens.append(tokenizer.eos_token_id)\n            count += 1\n            if len(tokens) >= cap:\n                break\n            if count % 500 == 0:\n                print(split, count, \'stories,\', len(tokens), \'tokens\', flush=True)\n        if len(tokens) < cap:\n            raise RuntimeError(f\'Insufficient {split} tokens: {len(tokens)} < {cap}\')\n        path = root / f\'{split}.npy\'\n        np.save(path, np.asarray(tokens[:cap], dtype=np.int32))\n        files[split] = {\'tokens\': cap, \'accepted_stories\': count, \'sha256\': digest(path)}\n        print(split, files[split], flush=True)\n    manifest = {\'spec\': spec, \'files\': files}\n    dump_json(manifest_path, manifest)\n    return manifest\n\n\ndef token_blocks(path, seq):\n    data = np.load(path, allow_pickle=False)\n    n = len(data) // (seq + 1)\n    if n < 2:\n        raise ValueError(\'Too few data blocks.\')\n    return torch.from_numpy(data[:n * (seq + 1)].reshape(n, seq + 1).astype(np.int64))\n\n\ndef training_batch(blocks, seed, step, micro, batch, device):\n    # Index generation is independent of model RNG, variant, interruptions and order.\n    generator = torch.Generator().manual_seed(seed * 1_000_003 + step * 10_007 + micro)\n    ids = torch.randint(len(blocks), (batch,), generator=generator)\n    packed = blocks[ids].to(device)\n    return packed[:, :-1], packed[:, 1:]\n\n\ndef amp_context(device, dtype):\n    return torch.autocast(\'cuda\', dtype=dtype) if device == \'cuda\' else contextlib.nullcontext()\n\n\ndef next_token_loss(model, x, y):\n    logits = model(x, use_cache=False).logits\n    return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), y.reshape(-1))\n\n\n@torch.no_grad()\ndef evaluate(model, blocks, batch, device, dtype, max_blocks):\n    model.eval()\n    total, tokens = 0.0, 0\n    for start in range(0, min(len(blocks), max_blocks), batch):\n        packed = blocks[start:min(start + batch, max_blocks)].to(device)\n        x, y = packed[:, :-1], packed[:, 1:]\n        with amp_context(device, dtype):\n            loss = next_token_loss(model, x, y)\n        total += loss.item() * y.numel()\n        tokens += y.numel()\n    model.train()\n    nll = total / tokens\n    return {\'validation_nll\': nll, \'validation_perplexity\': math.exp(nll),\n            \'validation_tokens\': tokens}\n\n\ndef synchronize(device):\n    if device == \'cuda\':\n        torch.cuda.synchronize()\n\n\ndef environment(device):\n    import transformers\n    return {\'python\': platform.python_version(), \'torch\': str(torch.__version__),\n            \'transformers\': transformers.__version__, \'cuda\': torch.version.cuda,\n            \'device\': device,\n            \'gpu\': torch.cuda.get_device_name(0) if device == \'cuda\' else None,\n            \'gpu_memory_bytes\': torch.cuda.get_device_properties(0).total_memory\n                if device == \'cuda\' else None}\n\n\ndef run_variant(args, variant, seed):\n    device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    if device == \'cpu\' and args.size != \'tiny\':\n        raise RuntimeError(\'GPU unavailable. Use --size tiny only for a CPU smoke test.\')\n    torch.set_num_threads(min(4, os.cpu_count() or 1))\n    dtype = torch.bfloat16 if device == \'cuda\' and torch.cuda.is_bf16_supported() else torch.float16\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    train = token_blocks(Path(args.data) / \'train.npy\', args.seq)\n    val = token_blocks(Path(args.data) / \'validation.npy\', args.seq)\n    run_dir = Path(args.output) / f\'{variant}_seed{seed}\'\n    run_dir.mkdir(parents=True, exist_ok=True)\n    spec = {k: getattr(args, k) for k in (\'size\', \'seq\', \'batch\', \'accum\', \'steps\',\n        \'lr\', \'eval_every\', \'eval_blocks\', \'block_sublayers\', \'save_every\')}\n    spec.update(seed=seed, variant=variant, upstream_commit=UPSTREAM,\n                data_manifest_sha256=digest(Path(args.data) / \'data_manifest.json\'),\n                experiment_sha256=digest(__file__), attnres_sha256=digest(Path(__file__).with_name(\'attnres.py\')),\n                environment=environment(device))\n    metadata = run_dir / \'config.json\'\n    if metadata.exists() and json.loads(metadata.read_text()) != spec:\n        raise RuntimeError(\'Run configuration/environment changed. Choose a new output folder.\')\n    dump_json(metadata, spec)\n    model = build_model(args.size, args.seq, variant, seed, args.block_sublayers).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=(0.9, 0.95), weight_decay=0.1)\n    scaler = torch.amp.GradScaler(\'cuda\', init_scale=1024.0,\n                                 enabled=device == \'cuda\' and dtype == torch.float16)\n    checkpoint = run_dir / \'latest.pt\'\n    history, timing, start = [], [], 0\n    if checkpoint.exists():\n        saved = torch.load(checkpoint, map_location=\'cpu\', weights_only=True)\n        if saved[\'spec\'] != spec:\n            raise RuntimeError(\'Checkpoint does not match this experiment.\')\n        model.load_state_dict(saved[\'model\'])\n        optimizer.load_state_dict(saved[\'optimizer\'])\n        scaler.load_state_dict(saved[\'scaler\'])\n        history, timing, start = saved[\'history\'], saved[\'timing\'], saved[\'step\']\n        torch.set_rng_state(saved[\'cpu_rng\'])\n        if device == \'cuda\':\n            torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        print(\'Resumed\', variant, \'at optimizer step\', start, flush=True)\n    else:\n        torch.manual_seed(seed + 1)\n        history.append({\'step\': 0, **evaluate(model, val, args.batch, device, dtype, args.eval_blocks)})\n    if device == \'cuda\':\n        torch.cuda.reset_peak_memory_stats()\n\n    def save(step):\n        state = {\'spec\': spec, \'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(),\n            \'scaler\': scaler.state_dict(), \'step\': step, \'history\': history, \'timing\': timing,\n            \'cpu_rng\': torch.get_rng_state(),\n            \'cuda_rng\': torch.cuda.get_rng_state_all() if device == \'cuda\' else []}\n        temp = checkpoint.with_suffix(\'.pt.tmp\')\n        torch.save(state, temp)\n        os.replace(temp, checkpoint)\n        dump_json(run_dir / \'history.json\', history)\n        dump_json(run_dir / \'timing.json\', timing)\n\n    print(variant, \'parameters:\', sum(p.numel() for p in model.parameters()), flush=True)\n    model.train()\n    for step in range(start, args.steps):\n        # Both variants receive the same warmup/cosine learning rate and token stream.\n        warmup = max(1, args.steps // 20)\n        progress = max(0, step - warmup) / max(1, args.steps - warmup)\n        lr = args.lr * min(1.0, (step + 1) / warmup) * (0.1 + 0.9 * (1 + math.cos(math.pi * progress)) / 2)\n        for group in optimizer.param_groups:\n            group[\'lr\'] = lr\n        optimizer.zero_grad(set_to_none=True)\n        synchronize(device)\n        began, losses = time.perf_counter(), []\n        for micro in range(args.accum):\n            x, y = training_batch(train, seed, step, micro, args.batch, device)\n            with amp_context(device, dtype):\n                loss = next_token_loss(model, x, y)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Non-finite loss. Last checkpoint retained.\')\n            losses.append(loss.detach())\n            scaler.scale(loss / args.accum).backward()\n        scaler.unscale_(optimizer)\n        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        if not torch.isfinite(grad_norm):\n            # No silently skipped FP16 updates: stop so paired token budgets stay equal.\n            raise FloatingPointError(\'Non-finite gradient. Reduce LR or adjust precision in a new paired run.\')\n        scaler.step(optimizer)\n        scaler.update()\n        synchronize(device)\n        elapsed = time.perf_counter() - began\n        timing.append({\'step\': step + 1, \'seconds\': elapsed,\n                       \'tokens\': args.batch * args.seq * args.accum})\n        if (step + 1) % 10 == 0 or step == start:\n            print(variant, seed, f\'{step+1}/{args.steps}\',\n                  \'train_nll\', round(torch.stack(losses).mean().item(), 4),\n                  \'tokens/s\', round(args.batch * args.seq * args.accum / elapsed), flush=True)\n        if (step + 1) % args.eval_every == 0 or step + 1 == args.steps:\n            row = {\'step\': step + 1, **evaluate(model, val, args.batch, device, dtype, args.eval_blocks)}\n            history.append(row)\n            print(\'EVAL\', variant, row, flush=True)\n        if (step + 1) % args.save_every == 0 or step + 1 == args.steps:\n            save(step + 1)\n    measured = [r for r in timing if r[\'step\'] > 2]\n    result = {\'variant\': variant, \'seed\': seed, \'completed_steps\': args.steps,\n              \'parameters\': sum(p.numel() for p in model.parameters()),\n              \'training_tokens\': args.steps * args.batch * args.seq * args.accum,\n              \'median_step_tokens_per_second\': statistics.median(r[\'tokens\'] / r[\'seconds\'] for r in measured)\n                  if measured else None,\n              \'training_step_seconds\': sum(r[\'seconds\'] for r in timing),\n              \'peak_allocated_gpu_bytes_this_session\': torch.cuda.max_memory_allocated() if device == \'cuda\' else None,\n              \'resume_step_this_session\': start,\n              \'environment\': environment(device), **history[-1]}\n    dump_json(run_dir / \'result.json\', result)\n    print(\'RESULT\', json.dumps(result), flush=True)\n    del model, optimizer, scaler\n    gc.collect()\n    if device == \'cuda\':\n        torch.cuda.empty_cache()\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'command\', choices=[\'prepare\', \'train\'])\n    parser.add_argument(\'--upstream\', default=\'vendor\')\n    parser.add_argument(\'--data\', default=\'data\')\n    parser.add_argument(\'--output\', default=\'runs/pilot001\')\n    parser.add_argument(\'--size\', choices=[\'tiny\', \'pilot\', \'standard\'], default=\'pilot\')\n    parser.add_argument(\'--seq\', type=int, default=128)\n    parser.add_argument(\'--batch\', type=int, default=8)\n    parser.add_argument(\'--accum\', type=int, default=4)\n    parser.add_argument(\'--steps\', type=int, default=200)\n    parser.add_argument(\'--lr\', type=float, default=3e-4)\n    parser.add_argument(\'--eval-every\', type=int, default=50)\n    parser.add_argument(\'--save-every\', type=int, default=25)\n    parser.add_argument(\'--eval-blocks\', type=int, default=256)\n    parser.add_argument(\'--block-sublayers\', type=int, default=4)\n    parser.add_argument(\'--seeds\', type=int, nargs=\'+\', default=[42])\n    parser.add_argument(\'--variants\', nargs=\'+\', choices=[\'baseline\', \'attnres\'], default=[\'baseline\', \'attnres\'])\n    args = parser.parse_args()\n    for key in (\'seq\', \'batch\', \'accum\', \'steps\', \'eval_every\', \'save_every\', \'eval_blocks\', \'block_sublayers\'):\n        if getattr(args, key) < 1:\n            parser.error(f\'{key} must be positive\')\n    if args.command == \'prepare\':\n        prepare_data(args.data, args.upstream)\n        return\n    results = []\n    for seed in args.seeds:\n        for variant in args.variants:\n            results.append(run_variant(args, variant, seed))\n    Path(args.output).mkdir(parents=True, exist_ok=True)\n    dump_json(Path(args.output) / \'results.json\', results)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'test_experiment.py': '"""Correctness checks with random inputs; these are not model quality results."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\n\nimport numpy as np\nimport torch\nfrom attnres import DepthMixer\nfrom experiment import build_model, next_token_loss, run_variant, training_batch\nimport experiment\n\n\nclass ExperimentTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(2)\n\n    def test_zero_query_means_uniform_depth_weights(self):\n        mixer = DepthMixer(8)\n        sources = [torch.randn(2, 5, 8) for _ in range(3)]\n        torch.testing.assert_close(mixer(sources), torch.stack(sources).mean(0))\n\n    def test_shared_initial_weights_identical(self):\n        a = build_model(\'tiny\', 16, \'baseline\', 42)\n        b = build_model(\'tiny\', 16, \'attnres\', 42)\n        for key, value in a.state_dict().items():\n            torch.testing.assert_close(value, b.state_dict()[key], rtol=0, atol=0)\n\n    def test_causal_outputs_and_gradients(self):\n        for block_size in (1, 3, 8):\n            model = build_model(\'tiny\', 16, \'attnres\', 42, block_size).eval()\n            x = torch.randint(10, 6400, (2, 12))\n            changed = x.clone()\n            changed[:, 7:] = torch.randint(10, 6400, changed[:, 7:].shape)\n            with torch.no_grad():\n                a, b = model(x).logits, model(changed).logits\n            torch.testing.assert_close(a[:, :7], b[:, :7], rtol=1e-5, atol=1e-5)\n            loss = next_token_loss(model, x[:, :-1], x[:, 1:])\n            self.assertTrue(torch.isfinite(loss))\n            loss.backward()\n            for name, parameter in model.named_parameters():\n                self.assertIsNotNone(parameter.grad, name)\n                self.assertTrue(torch.isfinite(parameter.grad).all(), name)\n            queries = [p.grad for n, p in model.named_parameters() if n.endswith(\'query\')]\n            self.assertTrue(any(g.abs().sum() > 0 for g in queries))\n\n    def test_batches_independent_of_model_rng(self):\n        data = torch.arange(20 * 9).reshape(20, 9)\n        a = training_batch(data, 42, 9, 2, 3, \'cpu\')\n        torch.randn(1000)\n        b = training_batch(data, 42, 9, 2, 3, \'cpu\')\n        for x, y in zip(a, b):\n            torch.testing.assert_close(x, y, rtol=0, atol=0)\n\n    def test_resuming_completed_run_preserves_result(self):\n        # Exercise saving/loading model, optimizer, scaler and RNG state end to end.\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            rng = np.random.default_rng(1)\n            for split in (\'train\', \'validation\'):\n                np.save(root / f\'{split}.npy\', rng.integers(0, 6400, 1000, dtype=np.int32))\n            (root / \'data_manifest.json\').write_text(json.dumps({\'synthetic_test\': True}))\n            args = argparse.Namespace(data=str(root), output=str(root / \'runs\'),\n                size=\'tiny\', seq=8, batch=2, accum=1, steps=3, lr=1e-4,\n                eval_every=3, eval_blocks=4, block_sublayers=3, save_every=1)\n            for variant in (\'baseline\', \'attnres\'):\n                first = run_variant(args, variant, 42)\n                second = run_variant(args, variant, 42)\n                self.assertEqual(first[\'validation_nll\'], second[\'validation_nll\'])\n                self.assertEqual(first[\'training_tokens\'], 48)\n                self.assertEqual(second[\'resume_step_this_session\'], 3)\n\n    def test_interrupted_training_matches_uninterrupted_cpu(self):\n        class SimulatedDisconnect(Exception):\n            pass\n        with tempfile.TemporaryDirectory() as tmp, patch(\'torch.cuda.is_available\', return_value=False):\n            root = Path(tmp)\n            rng = np.random.default_rng(2)\n            for split in (\'train\', \'validation\'):\n                np.save(root / f\'{split}.npy\', rng.integers(0, 6400, 1000, dtype=np.int32))\n            (root / \'data_manifest.json\').write_text(json.dumps({\'synthetic_test\': True}))\n            for variant in (\'baseline\', \'attnres\'):\n                args = argparse.Namespace(data=str(root), output=str(root / \'reference\'),\n                    size=\'tiny\', seq=8, batch=2, accum=1, steps=3, lr=1e-4,\n                    eval_every=3, eval_blocks=4, block_sublayers=3, save_every=1)\n                reference = run_variant(args, variant, 42)\n                original_dump = experiment.dump_json\n                def interrupt_after_checkpoint(path, value):\n                    original_dump(path, value)\n                    if Path(path).name == \'history.json\':\n                        raise SimulatedDisconnect()\n                args.output = str(root / \'interrupted\')\n                with patch(\'experiment.dump_json\', side_effect=interrupt_after_checkpoint):\n                    with self.assertRaises(SimulatedDisconnect):\n                        run_variant(args, variant, 42)\n                resumed = run_variant(args, variant, 42)\n                self.assertEqual(resumed[\'resume_step_this_session\'], 1)\n                self.assertAlmostEqual(reference[\'validation_nll\'], resumed[\'validation_nll\'], places=6)\n                a = torch.load(root / \'reference\' / f\'{variant}_seed42/latest.pt\', weights_only=True)\n                b = torch.load(root / \'interrupted\' / f\'{variant}_seed42/latest.pt\', weights_only=True)\n                for key, value in a[\'model\'].items():\n                    torch.testing.assert_close(value, b[\'model\'][key], rtol=0, atol=1e-7)\n\n\nif __name__ == \'__main__\':\n    unittest.main(verbosity=2)\n', 'phase2/optimized.py': '"""Training-only wrappers and a cached-RMS implementation of Block AttnRes.\n\nOriginal phase-one files remain unchanged. No KV-cache generation is implemented.\n"""\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom experiment import build_model\n\n\nclass TrainingLM(nn.Module):\n    def __init__(self, original, mode):\n        super().__init__()\n        self.backbone = original.model\n        self.lm_head = original.lm_head\n        self.mode = mode\n        if mode not in (\'upstream\', \'fp32\', \'attnres_reference\', \'attnres_cached\'):\n            raise ValueError(mode)\n\n    @staticmethod\n    def inverse_rms(values, eps):\n        return torch.rsqrt(values.square().mean(-1, keepdim=True) + eps)\n\n    @staticmethod\n    def mix_cached(mixer, completed, inverse, partial):\n        if completed.shape[0] == 1 and partial is None:\n            # A softmax with one source is exactly one. These query/norm\n            # parameters are mathematically inactive and may have grad=None.\n            return completed[0]\n        effective_query = mixer.query.float() * mixer.key_norm.weight.float()\n        scores = (completed * effective_query).sum(-1) * inverse.squeeze(-1)\n        if partial is not None:\n            p = partial.float()\n            inv_p = TrainingLM.inverse_rms(p, mixer.key_norm.eps)\n            p_score = (p * effective_query).sum(-1) * inv_p.squeeze(-1)\n            weights = torch.cat((scores, p_score.unsqueeze(0)), dim=0).softmax(0)\n            return (completed * weights[:-1].unsqueeze(-1)).sum(0) + p * weights[-1].unsqueeze(-1)\n        weights = scores.softmax(0)\n        return (completed * weights.unsqueeze(-1)).sum(0)\n\n    def forward(self, x):\n        body = self.backbone\n        if self.mode == \'attnres_reference\':\n            hidden, _, _ = body(x, use_cache=False)\n            return self.lm_head(hidden)\n        embedding = body.dropout(body.embed_tokens(x))\n        positions = (body.freqs_cos[:x.shape[1]], body.freqs_sin[:x.shape[1]])\n        if self.mode in (\'upstream\', \'fp32\'):\n            hidden = embedding\n            for layer in body.layers:\n                delta, _ = layer.self_attn(layer.input_layernorm(hidden), positions, use_cache=False)\n                if self.mode == \'upstream\':\n                    # Preserve upstream\'s in-place destination dtype exactly.\n                    hidden = delta.add_(hidden)\n                else:\n                    hidden = hidden + delta.float()\n                delta = layer.mlp(layer.post_attention_layernorm(hidden))\n                hidden = hidden + (delta.float() if self.mode == \'fp32\' else delta)\n            return self.lm_head(body.norm(hidden))\n        completed = embedding.unsqueeze(0).float()\n        inverse = self.inverse_rms(completed, body.config.rms_norm_eps)\n        partial, index = None, 0\n        for layer in body.layers:\n            for is_attention in (True, False):\n                hidden = self.mix_cached(body.depth_mixers[index], completed, inverse, partial)\n                if is_attention:\n                    delta, _ = layer.self_attn(layer.input_layernorm(hidden), positions, use_cache=False)\n                else:\n                    delta = layer.mlp(layer.post_attention_layernorm(hidden))\n                # Match phase-one partial-sum precision; do not silently make\n                # this a different architecture/precision experiment.\n                partial = delta if partial is None else partial + delta\n                index += 1\n                if index % body.block_sublayers == 0:\n                    value = partial.float().unsqueeze(0)\n                    completed = torch.cat((completed, value), dim=0)\n                    inverse = torch.cat((inverse, self.inverse_rms(value, body.config.rms_norm_eps)), dim=0)\n                    partial = None\n        hidden = self.mix_cached(body.depth_mixers[index], completed, inverse, partial)\n        return self.lm_head(body.norm(hidden))\n\n\nclass TrainingLoss(nn.Module):\n    def __init__(self, model):\n        super().__init__()\n        self.model = model\n\n    def forward(self, x, y):\n        logits = self.model(x)\n        return F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), y.reshape(-1))\n\n\ndef create_model(mode, size=\'pilot\', seq=128, seed=42, block_sublayers=4):\n    variant = \'attnres\' if mode.startswith(\'attnres\') else \'baseline\'\n    original = build_model(size, seq, variant, seed, block_sublayers)\n    return TrainingLM(original, mode)\n', 'phase2/controlled.py': '"""Both comparison models use FP32 residual states, including AttnRes block sums."""\nimport torch\nfrom optimized import TrainingLM\nfrom experiment import build_model\n\n\nclass ControlledLM(TrainingLM):\n    def __init__(self,original,architecture):\n        super().__init__(original, \'fp32\' if architecture==\'baseline\' else \'attnres_cached\')\n        self.architecture=architecture\n\n    def forward(self,x):\n        if self.architecture==\'baseline\':\n            return super().forward(x)\n        body=self.backbone\n        embedding=body.dropout(body.embed_tokens(x)).float()\n        positions=(body.freqs_cos[:x.shape[1]],body.freqs_sin[:x.shape[1]])\n        completed=embedding.unsqueeze(0)\n        inverse=self.inverse_rms(completed,body.config.rms_norm_eps)\n        partial,index=None,0\n        for layer in body.layers:\n            for is_attention in (True,False):\n                hidden=self.mix_cached(body.depth_mixers[index],completed,inverse,partial)\n                if is_attention:\n                    delta,_=layer.self_attn(layer.input_layernorm(hidden),positions,use_cache=False)\n                else:\n                    delta=layer.mlp(layer.post_attention_layernorm(hidden))\n                delta=delta.float()\n                partial=delta if partial is None else partial+delta\n                index+=1\n                if index%body.block_sublayers==0:\n                    value=partial.unsqueeze(0)\n                    completed=torch.cat((completed,value),0)\n                    inverse=torch.cat((inverse,self.inverse_rms(value,body.config.rms_norm_eps)),0)\n                    partial=None\n        hidden=self.mix_cached(body.depth_mixers[index],completed,inverse,partial)\n        return self.lm_head(body.norm(hidden))\n\n\ndef controlled_model(architecture,seed=42):\n    original=build_model(\'pilot\',128,architecture,seed,4)\n    return ControlledLM(original,architecture)\n', 'phase2/test_optimized.py': '"""Equivalence checks for numerical changes and cached depth mixing."""\nimport copy\nimport json\nfrom pathlib import Path\nimport unittest\nimport torch\nfrom experiment import build_model\nfrom optimized import TrainingLM, TrainingLoss\n\n\nclass OptimizedTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(2)\n\n    def test_upstream_wrapper_matches_original(self):\n        original = build_model(\'tiny\',16,\'baseline\',42).eval()\n        wrapped = TrainingLM(copy.deepcopy(original), \'upstream\').eval()\n        x = torch.randint(0,6400,(2,12))\n        torch.testing.assert_close(original(x).logits, wrapped(x), rtol=0, atol=0)\n\n    def test_cached_logits_and_gradients(self):\n        for block in (1,3,4,8):\n            original = build_model(\'tiny\',16,\'attnres\',42,block)\n            # Nonzero queries and learned RMS weights exercise actual depth selection.\n            with torch.no_grad():\n                for m in original.model.depth_mixers:\n                    m.query.normal_(0,.1)\n                    m.key_norm.weight.uniform_(.8,1.2)\n            a = TrainingLM(copy.deepcopy(original),\'attnres_reference\')\n            b = TrainingLM(copy.deepcopy(original),\'attnres_cached\')\n            x = torch.randint(0,6400,(2,12))\n            y = torch.randint(0,6400,(2,12))\n            torch.testing.assert_close(a(x), b(x), rtol=2e-4, atol=2e-6)\n            TrainingLoss(a)(x,y).backward()\n            TrainingLoss(b)(x,y).backward()\n            for (name,p),(name2,q) in zip(a.named_parameters(),b.named_parameters()):\n                self.assertEqual(name,name2)\n                gp = torch.zeros_like(p) if p.grad is None else p.grad\n                gq = torch.zeros_like(q) if q.grad is None else q.grad\n                torch.testing.assert_close(gp,gq,rtol=2e-3,atol=3e-6,msg=name)\n\n    def test_cached_is_causal(self):\n        model=TrainingLM(build_model(\'tiny\',16,\'attnres\',42),\'attnres_cached\').eval()\n        x=torch.randint(0,6400,(2,12))\n        y=x.clone(); y[:,7:]=torch.randint(0,6400,y[:,7:].shape)\n        torch.testing.assert_close(model(x)[:,:7],model(y)[:,:7],rtol=1e-5,atol=1e-5)\n\n    @unittest.skipUnless(torch.cuda.is_available(),\'CUDA check runs in Colab\')\n    def test_cuda_fp16_equivalence(self):\n        original=build_model(\'pilot\',128,\'attnres\',42)\n        with torch.no_grad():\n            for mixer in original.model.depth_mixers:\n                mixer.query.normal_(0,.02)\n        a=TrainingLM(copy.deepcopy(original),\'attnres_reference\').cuda()\n        b=TrainingLM(copy.deepcopy(original),\'attnres_cached\').cuda()\n        x=torch.randint(0,6400,(2,128),device=\'cuda\')\n        with torch.autocast(\'cuda\',dtype=torch.float16):\n            ya,yb=a(x),b(x)\n        torch.testing.assert_close(ya,yb,rtol=1e-2,atol=2e-3)\n        # The wrapper must also preserve upstream\'s mixed-precision path.\n        original=build_model(\'tiny\',16,\'baseline\',42).cuda()\n        wrapped=TrainingLM(copy.deepcopy(original),\'upstream\').cuda()\n        with torch.autocast(\'cuda\',dtype=torch.float16):\n            ya=original(x[:,:12]).logits; yb=wrapped(x[:,:12])\n        torch.testing.assert_close(ya,yb,rtol=0,atol=0)\n\n\nif __name__==\'__main__\':\n    unittest.main(verbosity=2)\n', 'phase2/test_controlled.py': '"""Check shared initialization, precision, causal masking and compilation."""\nimport copy\nimport gc\nimport json\nfrom pathlib import Path\nimport time\nimport unittest\nimport torch\nfrom experiment import build_model\nfrom controlled import ControlledLM,controlled_model\nfrom optimized import TrainingLM,TrainingLoss\n\n\nclass ControlledTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):torch.set_num_threads(2)\n\n    def test_shared_initialization(self):\n        a=controlled_model(\'baseline\').state_dict()\n        b=controlled_model(\'attnres\').state_dict()\n        for key,value in a.items():torch.testing.assert_close(value,b[key],rtol=0,atol=0)\n\n    def test_reference_math_and_causality(self):\n        original=build_model(\'tiny\',16,\'attnres\',42)\n        with torch.no_grad():\n            for m in original.model.depth_mixers:\n                m.query.normal_(0,.1);m.key_norm.weight.uniform_(.8,1.2)\n        a=TrainingLM(copy.deepcopy(original),\'attnres_reference\')\n        b=ControlledLM(copy.deepcopy(original),\'attnres\')\n        x=torch.randint(0,6400,(2,12));y=x.clone();y[:,7:]=0\n        torch.testing.assert_close(a(x),b(x),rtol=2e-4,atol=2e-6)\n        torch.testing.assert_close(b(x)[:,:7],b(y)[:,:7],rtol=1e-5,atol=1e-5)\n        TrainingLoss(a)(x,y).backward();TrainingLoss(b)(x,y).backward()\n        for (name,p),(_,q) in zip(a.named_parameters(),b.named_parameters()):\n            gp=torch.zeros_like(p) if p.grad is None else p.grad\n            gq=torch.zeros_like(q) if q.grad is None else q.grad\n            self.assertTrue(torch.isfinite(gq).all(),name)\n            torch.testing.assert_close(gp,gq,rtol=2e-3,atol=3e-6,msg=name)\n\n    @unittest.skipUnless(torch.cuda.is_available(),\'GPU test runs in Colab\')\n    def test_fp32_states_under_autocast(self):\n        for architecture in (\'baseline\',\'attnres\'):\n            model=controlled_model(architecture).cuda()\n            seen=[]\n            hooks=[norm.register_forward_pre_hook(lambda m,args:seen.append(args[0].dtype))\n                for layer in model.backbone.layers for norm in (layer.input_layernorm,layer.post_attention_layernorm)]\n            with torch.autocast(\'cuda\',dtype=torch.float16):model(torch.ones((2,128),dtype=torch.long,device=\'cuda\'))\n            self.assertEqual(len(seen),12)\n            self.assertTrue(all(dtype==torch.float32 for dtype in seen))\n            for hook in hooks:hook.remove()\n            del model;gc.collect();torch.cuda.empty_cache()\n\n\ndef compiled_checks():\n    rows=[]\n    for architecture in (\'baseline\',\'attnres\'):\n        eager=TrainingLoss(controlled_model(architecture).cuda())\n        candidate=copy.deepcopy(eager)\n        compiled=torch.compile(candidate,fullgraph=True,dynamic=False,mode=\'default\')\n        generator=torch.Generator(device=\'cuda\').manual_seed(7)\n        x=torch.randint(0,6400,(32,128),generator=generator,device=\'cuda\')\n        y=torch.randint(0,6400,(32,128),generator=generator,device=\'cuda\')\n        began=time.perf_counter()\n        with torch.autocast(\'cuda\',dtype=torch.float16):a=eager(x,y);b=compiled(x,y)\n        # Match the actual mixed-precision trainer: scale before backward.\n        # Unscaled FP16 CE gradients can underflow before reaching FP32 weights.\n        scale=1024.\n        (a*scale).backward();(b*scale).backward();torch.cuda.synchronize()\n        for objective in (eager,candidate):\n            for parameter in objective.parameters():\n                if parameter.grad is not None:parameter.grad.div_(scale)\n        torch.testing.assert_close(a,b,rtol=1e-3,atol=1e-3)\n        max_error=0.;details=[]\n        for (name,p),(_,q) in zip(eager.named_parameters(),candidate.named_parameters()):\n            gp=torch.zeros_like(p) if p.grad is None else p.grad\n            gq=torch.zeros_like(q) if q.grad is None else q.grad\n            self_max=(gp-gq).abs().max().item();max_error=max(max_error,self_max)\n            details.append({\'parameter\':name,\'max_abs_error\':self_max,\'eager_max_abs\':gp.abs().max().item(),\n                \'eager_norm\':gp.norm().item(),\'difference_norm\':(gp-gq).norm().item()})\n        diagnostic={\'architecture\':architecture,\'gradient_scale\':scale,\'eager_loss\':a.item(),\'compiled_loss\':b.item(),\'parameters\':details}\n        diagnostics=Path(__file__).parent/\'results\'/f\'compiled_diagnostic_{architecture}.json\'\n        diagnostics.write_text(json.dumps(diagnostic,indent=2))\n        print(\'COMPILED_DIAGNOSTIC\',json.dumps({\'architecture\':architecture,\'eager_loss\':a.item(),\'compiled_loss\':b.item(),\n            \'largest_errors\':sorted(details,key=lambda r:r[\'max_abs_error\'],reverse=True)[:5]}),flush=True)\n        passed=True\n        for (_,p),(_,q) in zip(eager.named_parameters(),candidate.named_parameters()):\n            gp=torch.zeros_like(p) if p.grad is None else p.grad\n            gq=torch.zeros_like(q) if q.grad is None else q.grad\n            try:torch.testing.assert_close(gp,gq,rtol=.04,atol=2e-4)\n            except AssertionError:passed=False\n        row={\'architecture\':architecture,\'loss_absolute_error\':abs(a.item()-b.item()),\n            \'max_gradient_absolute_error\':max_error,\'seconds_including_compile\':time.perf_counter()-began,\n            \'gradient_rtol\':.04,\'gradient_atol\':2e-4,\'loss_rtol\':1e-3,\'loss_atol\':1e-3,\n            \'gradient_scale\':scale,\n            \'status\':\'passed\' if passed else \'failed\'}\n        rows.append(row);print(\'COMPILED_EQUIVALENCE\',json.dumps(row),flush=True)\n        del eager,candidate,compiled,a,b;gc.collect();torch.cuda.empty_cache()\n    path=Path(__file__).parent/\'results/controlled_compilation_checks.json\'\n    path.parent.mkdir(exist_ok=True);path.write_text(json.dumps(rows,indent=2))\n\n\nif __name__==\'__main__\':\n    suite=unittest.defaultTestLoader.loadTestsFromTestCase(ControlledTests)\n    outcome=unittest.TextTestRunner(verbosity=2).run(suite)\n    if not outcome.wasSuccessful():raise SystemExit(1)\n    if torch.cuda.is_available():compiled_checks()\n', 'phase2/benchmark_stage2.py': '"""Isolated bounded trials: no quality claims from these throughput measurements."""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport statistics\nimport subprocess\nimport sys\nimport time\nimport torch\nfrom torch import nn\nfrom experiment import build_model, environment, token_blocks, training_batch\nfrom optimized import create_model, TrainingLoss\n\nROOT=Path(__file__).resolve().parent.parent\nOUT=ROOT/\'phase2/results\'\nOUT.mkdir(exist_ok=True,parents=True)\nCONFIGS=[\n    {\'name\':\'original_baseline_8x4\',\'mode\':\'baseline_original\',\'physical_batch\':8,\'compile\':False,\'fused\':False},\n    {\'name\':\'original_attnres_8x4\',\'mode\':\'attnres_original\',\'physical_batch\':8,\'compile\':False,\'fused\':False},\n    {\'name\':\'fp32_baseline_eager_32\',\'mode\':\'fp32\',\'physical_batch\':32,\'compile\':False,\'fused\':True},\n    {\'name\':\'cached_attnres_eager_32\',\'mode\':\'attnres_cached\',\'physical_batch\':32,\'compile\':False,\'fused\':True},\n    {\'name\':\'fp32_baseline_compiled_32\',\'mode\':\'fp32\',\'physical_batch\':32,\'compile\':True,\'fused\':True},\n    {\'name\':\'cached_attnres_compiled_32\',\'mode\':\'attnres_cached\',\'physical_batch\':32,\'compile\':True,\'fused\':True}]\n\nclass OriginalLogits(nn.Module):\n    def __init__(self,model):\n        super().__init__(); self.original=model\n    def forward(self,x):\n        return self.original(x,use_cache=False).logits\n\ndef logical_batch(blocks,seed,step,device=\'cuda\'):\n    # Same four logical groups of eight as phase one, merged before transfer.\n    parts=[training_batch(blocks,seed,step,m,8,\'cpu\') for m in range(4)]\n    return tuple(torch.cat([p[k] for p in parts],0).to(device) for k in (0,1))\n\ndef trial(config):\n    assert torch.cuda.is_available()\n    torch.set_num_threads(4)\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n    torch.cuda.reset_peak_memory_stats()\n    if config[\'mode\'].endswith(\'_original\'):\n        variant=\'baseline\' if config[\'mode\'].startswith(\'baseline\') else \'attnres\'\n        model=OriginalLogits(build_model(\'pilot\',128,variant,42)).cuda()\n    else:\n        model=create_model(config[\'mode\']).cuda()\n    objective=TrainingLoss(model)\n    if config[\'compile\']:\n        objective=torch.compile(objective,fullgraph=True,dynamic=False,mode=\'default\')\n    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,betas=(.9,.95),weight_decay=.1,fused=config[\'fused\'])\n    scaler=torch.amp.GradScaler(\'cuda\',init_scale=1024)\n    data=token_blocks(ROOT/\'data/train.npy\',128)\n    timings=[]\n    first_step=None\n    for index in range(68):\n        optimizer.zero_grad(set_to_none=True)\n        torch.cuda.synchronize(); start=time.perf_counter()\n        x,y=logical_batch(data,42,index)\n        for lo in range(0,32,config[\'physical_batch\']):\n            hi=lo+config[\'physical_batch\']\n            with torch.autocast(\'cuda\',dtype=torch.float16):\n                loss=objective(x[lo:hi],y[lo:hi]) * (config[\'physical_batch\']/32)\n            scaler.scale(loss).backward()\n        scaler.unscale_(optimizer)\n        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True)\n        scaler.step(optimizer); scaler.update()\n        torch.cuda.synchronize(); elapsed=time.perf_counter()-start\n        if index==0:\n            first_step=elapsed\n            print(\'FIRST_STEP\',config[\'name\'],round(elapsed,3),flush=True)\n        if index>=8:\n            timings.append(elapsed)\n    row={\'config\':config,\'environment\':environment(\'cuda\'),\n         \'first_step_seconds_including_cold_compile\':first_step,\n         \'warmup_steps\':8,\'measured_steps\':60,\'logical_batch\':32,\'sequence\':128,\n         \'median_training_tokens_per_second\':4096/statistics.median(timings),\n         \'p10_step_seconds\':sorted(timings)[6], \'p90_step_seconds\':sorted(timings)[53],\n         \'peak_allocated_gpu_bytes\':torch.cuda.max_memory_allocated(),\n         \'step_seconds\':timings,\'status\':\'ok\'}\n    (OUT/f"benchmark_{config[\'name\']}.json").write_text(json.dumps(row,indent=2))\n    print(\'BENCHMARK\',json.dumps({k:v for k,v in row.items() if k!=\'step_seconds\'}),flush=True)\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--trial\',type=int)\n    args=parser.parse_args()\n    if args.trial is not None:\n        trial(CONFIGS[args.trial])\n    else:\n        rows=[]\n        for index,config in enumerate(CONFIGS):\n            print(\'START_TRIAL\',config[\'name\'],flush=True)\n            try:\n                done=subprocess.run([sys.executable,__file__,\'--trial\',str(index)],timeout=240)\n                path=OUT/f"benchmark_{config[\'name\']}.json"\n                if done.returncode==0 and path.exists():\n                    rows.append(json.loads(path.read_text()))\n                else:\n                    rows.append({\'config\':config,\'status\':\'failed\',\'returncode\':done.returncode})\n            except subprocess.TimeoutExpired:\n                rows.append({\'config\':config,\'status\':\'timeout_240_seconds\'})\n            (OUT/\'benchmark_summary.json\').write_text(json.dumps(rows,indent=2))\n        print(\'BENCHMARKS_COMPLETE\',flush=True)\n', 'phase2/prepare_holdout.py': '"""Create an untouched story-level holdout after the phase-one validation prefix."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom datasets import load_dataset\nfrom transformers import AutoTokenizer\nfrom experiment import DATASET, DATA_REVISION, digest, dump_json\n\nROOT=Path(__file__).resolve().parent.parent\nOUT=ROOT/\'phase2/data\'\nOUT.mkdir(parents=True,exist_ok=True)\nif (OUT/\'holdout_manifest.json\').exists():\n    manifest=json.loads((OUT/\'holdout_manifest.json\').read_text())\n    assert digest(OUT/\'holdout.npy\')==manifest[\'holdout_sha256\']\n    print(\'HOLDOUT_CACHE_VERIFIED\',manifest,flush=True)\n    raise SystemExit(0)\nold=json.loads((ROOT/\'data/data_manifest.json\').read_text())\nfor split in (\'train\',\'validation\'):\n    assert digest(ROOT/f\'data/{split}.npy\')==old[\'files\'][split][\'sha256\']\ntokenizer=AutoTokenizer.from_pretrained(str(ROOT/\'vendor/model\'),trust_remote_code=False)\nseen=set()\ndef encode(row):\n    text=row[\'text\'].strip()\n    if not text:\n        return None\n    key=hashlib.sha256(text.encode(\'utf-8\')).digest()\n    if key in seen:\n        return None\n    seen.add(key)\n    return tokenizer.encode(text,add_special_tokens=False)+[tokenizer.eos_token_id]\ntrain_tokens=[]; train_count=0\nfor row in load_dataset(DATASET,revision=DATA_REVISION,split=\'train\',streaming=True):\n    token_ids=encode(row)\n    if token_ids is None:\n        continue\n    train_tokens.extend(token_ids);train_count+=1\n    if len(train_tokens)>=old[\'files\'][\'train\'][\'tokens\']:\n        break\nassert np.array_equal(np.asarray(train_tokens[:2_000_000],dtype=np.int32),np.load(ROOT/\'data/train.npy\'))\nprint(\'Original train prefix reconstructed exactly; complete-story hashes retained.\',flush=True)\nvalidation_tokens=[];holdout=[];prefix_count=0;holdout_count=0\nfor row in load_dataset(DATASET,revision=DATA_REVISION,split=\'validation\',streaming=True):\n    token_ids=encode(row)\n    if token_ids is None:\n        continue\n    if len(validation_tokens)<100_000:\n        validation_tokens.extend(token_ids);prefix_count+=1\n    else:\n        holdout.extend(token_ids);holdout_count+=1\n        if len(holdout)>=128_000:\n            break\nassert np.array_equal(np.asarray(validation_tokens[:100_000],dtype=np.int32),np.load(ROOT/\'data/validation.npy\'))\nassert len(holdout)>=128_000\nnp.save(OUT/\'holdout.npy\',np.asarray(holdout[:128_000],dtype=np.int32))\nmanifest={\'dataset\':DATASET,\'revision\':DATA_REVISION,\n    \'selection\':\'Subsequent complete stories from official validation split, after all phase-one validation stories; exact text dedup against complete selected train/validation stories\',\n    \'train_stories_reconstructed\':train_count,\'validation_prefix_stories\':prefix_count,\n    \'holdout_stories\':holdout_count,\'holdout_token_cap\':128_000,\n    \'holdout_sha256\':digest(OUT/\'holdout.npy\'),\n    \'phase1_data_manifest_sha256\':digest(ROOT/\'data/data_manifest.json\'),\n    \'role\':\'Held-out evaluation partition; not the dataset official test split; do not use for selecting hyperparameters.\'}\ndump_json(OUT/\'holdout_manifest.json\',manifest)\nprint(\'HOLDOUT_READY\',json.dumps(manifest),flush=True)\n', 'phase2/train_stage2.py': '"""Longer, precision-controlled training; heldout scores only at the fixed endpoint."""\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport statistics\nimport time\nimport torch\nfrom experiment import dump_json,digest,environment,token_blocks\nfrom benchmark_stage2 import logical_batch\nfrom optimized import TrainingLoss\nfrom controlled import controlled_model\nfrom backup import backup_tree\n\nROOT=Path(__file__).resolve().parent.parent\nHERE=ROOT/\'phase2\'\nOUT=HERE/\'runs/controlled1000\'\nOUT.mkdir(parents=True,exist_ok=True)\nBACKUP=Path(os.environ[\'MINIK3_DRIVE_BACKUP\']) if os.environ.get(\'MINIK3_DRIVE_BACKUP\') else None\nassert torch.cuda.is_available()\ntorch.set_num_threads(4)\ntorch.backends.cuda.matmul.allow_tf32=False\ntorch.backends.cudnn.allow_tf32=False\nbench=json.loads((HERE/\'results/benchmark_summary.json\').read_text())\nby_name={r[\'config\'][\'name\']:r for r in bench}\ncompiled_names=(\'fp32_baseline_compiled_32\',\'cached_attnres_compiled_32\')\ncompile_checks=json.loads((HERE/\'results/controlled_compilation_checks.json\').read_text())\nuse_compile=len(compile_checks)==2 and all(r[\'status\']==\'passed\' for r in compile_checks)\nuse_compile=use_compile and all(by_name.get(name,{}).get(\'status\')==\'ok\' for name in compiled_names)\nif use_compile:\n    use_compile=all(by_name[c][\'median_training_tokens_per_second\'] > 1.05*by_name[e][\'median_training_tokens_per_second\']\n        for c,e in zip(compiled_names,(\'fp32_baseline_eager_32\',\'cached_attnres_eager_32\')))\nprint(\'SELECTED_ENGINE\', \'inductor\' if use_compile else \'eager\',flush=True)\ndata=token_blocks(ROOT/\'data/train.npy\',128)\nvalidation=token_blocks(ROOT/\'data/validation.npy\',128)[:256]\nholdout=token_blocks(HERE/\'data/holdout.npy\',128)\nsource_hashes={name:digest(HERE/name) for name in [\'optimized.py\',\'controlled.py\',\'train_stage2.py\',\'benchmark_stage2.py\',\'backup.py\',\'prepare_holdout.py\',\'../experiment.py\',\'../attnres.py\']}\n\n@torch.no_grad()\ndef evaluate(model,blocks):\n    model.eval(); total,tokens=0.,0\n    loss_fn=TrainingLoss(model)\n    for lo in range(0,len(blocks),32):\n        pack=blocks[lo:lo+32].cuda()\n        with torch.autocast(\'cuda\',dtype=torch.float16):\n            loss=loss_fn(pack[:,:-1],pack[:,1:])\n        count=pack.shape[0]*128\n        total+=loss.item()*count; tokens+=count\n    model.train()\n    return {\'nll\':total/tokens,\'perplexity\':math.exp(total/tokens),\'tokens\':tokens}\n\n\ndef train(architecture,seed):\n    start_wall=time.perf_counter()\n    folder=OUT/f\'{architecture}_seed{seed}\'\n    folder.mkdir(exist_ok=True)\n    spec={\'architecture\':architecture,\'seed\':seed,\'steps\':1000,\'sequence\':128,\n        \'physical_batch\':32,\'logical_batch\':32,\'lr\':3e-4,\'optimizer\':\'AdamW fused\',\n        \'betas\':[.9,.95],\'weight_decay\':.1,\'gradient_clip\':1.,\'residual_dtype\':\'float32\',\n        \'linear_autocast\':\'float16\',\'engine\':\'inductor\' if use_compile else \'eager\',\n        \'source_hashes\':source_hashes,\'data_manifest_sha256\':digest(ROOT/\'data/data_manifest.json\'),\n        \'compilation_checks_sha256\':digest(HERE/\'results/controlled_compilation_checks.json\'),\n        \'holdout_manifest_sha256\':digest(HERE/\'data/holdout_manifest.json\'),\n        \'environment\':environment(\'cuda\')}\n    if (folder/\'config.json\').exists():\n        assert json.loads((folder/\'config.json\').read_text())==spec,\'Use a new run directory after changing settings or source.\'\n    dump_json(folder/\'config.json\',spec)\n    if (folder/\'result.json\').exists():\n        print(\'ALREADY_COMPLETE\',architecture,seed,flush=True)\n        return json.loads((folder/\'result.json\').read_text())\n    model=controlled_model(architecture,seed).cuda().train()\n    objective=TrainingLoss(model)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,betas=(.9,.95),weight_decay=.1,fused=True)\n    scaler=torch.amp.GradScaler(\'cuda\',init_scale=1024)\n    history,timing,start=[],[],0\n    checkpoint=folder/\'latest.pt\'\n    if checkpoint.exists():\n        saved=torch.load(checkpoint,map_location=\'cpu\',weights_only=True)\n        assert saved[\'spec\']==spec\n        model.load_state_dict(saved[\'model\']);optimizer.load_state_dict(saved[\'optimizer\'])\n        scaler.load_state_dict(saved[\'scaler\']);history=saved[\'history\'];timing=saved[\'timing\'];start=saved[\'step\']\n        torch.set_rng_state(saved[\'cpu_rng\']);torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        print(\'RESUME\',architecture,seed,start,flush=True)\n    else:\n        history.append({\'step\':0,\'validation\':evaluate(model,validation)})\n    if use_compile:\n        objective=torch.compile(objective,fullgraph=True,dynamic=False,mode=\'default\')\n    torch.cuda.reset_peak_memory_stats()\n    def save(step):\n        state={\'spec\':spec,\'step\':step,\'model\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\n            \'scaler\':scaler.state_dict(),\'history\':history,\'timing\':timing,\n            \'cpu_rng\':torch.get_rng_state(),\'cuda_rng\':torch.cuda.get_rng_state_all()}\n        temporary=checkpoint.with_suffix(\'.pt.tmp\')\n        torch.save(state,temporary);os.replace(temporary,checkpoint)\n        dump_json(folder/\'history.json\',history);dump_json(folder/\'timing.json\',timing)\n        if BACKUP and step%500==0:\n            backup_tree(folder,BACKUP/folder.relative_to(ROOT))\n    for step in range(start,1000):\n        lr=3e-4*min(1.,(step+1)/50)*(.1+.9*(1+math.cos(math.pi*max(0,step-50)/950))/2)\n        for group in optimizer.param_groups:group[\'lr\']=lr\n        optimizer.zero_grad(set_to_none=True)\n        torch.cuda.synchronize();began=time.perf_counter()\n        x,y=logical_batch(data,seed,step)\n        with torch.autocast(\'cuda\',dtype=torch.float16):loss=objective(x,y)\n        scaler.scale(loss).backward();scaler.unscale_(optimizer)\n        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True)\n        if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite loss\')\n        scaler.step(optimizer);scaler.update()\n        torch.cuda.synchronize();elapsed=time.perf_counter()-began\n        timing.append({\'step\':step+1,\'seconds\':elapsed,\'tokens\':4096})\n        if (step+1)%100==0 or step==start:\n            print(\'TRAIN\',architecture,seed,step+1,\'nll\',round(loss.item(),4),\'tokens/s\',round(4096/elapsed),flush=True)\n        if (step+1)%100==0:\n            row={\'step\':step+1,\'validation\':evaluate(model,validation)}\n            history.append(row);print(\'VALIDATION\',architecture,seed,json.dumps(row),flush=True)\n            save(step+1)\n    # Endpoint-only evaluation: no hyperparameter selection against this score.\n    heldout=evaluate(model,holdout)\n    measured=[r for r in timing if r[\'step\']>8]\n    result={\'architecture\':architecture,\'seed\':seed,\'steps\':1000,\'training_tokens\':4_096_000,\n        \'parameters\':sum(p.numel() for p in model.parameters()),\n        \'validation\':history[-1][\'validation\'],\'heldout\':heldout,\n        \'median_training_tokens_per_second\':statistics.median(4096/r[\'seconds\'] for r in measured),\n        \'first_step_seconds_including_cold_compile\':timing[0][\'seconds\'],\n        \'training_step_seconds\':sum(r[\'seconds\'] for r in timing),\n        \'run_wall_seconds_including_compile_eval_checkpoint_io\':time.perf_counter()-start_wall,\n        \'peak_allocated_gpu_bytes_this_session\':torch.cuda.max_memory_allocated(),\n        \'resume_step_this_session\':start,\'engine\':spec[\'engine\'],\'environment\':spec[\'environment\']}\n    dump_json(folder/\'result.json\',result)\n    if BACKUP:\n        backup_tree(folder,BACKUP/folder.relative_to(ROOT))\n    print(\'CONTROLLED_RESULT\',json.dumps(result),flush=True)\n    del model,objective,optimizer,scaler\n    gc.collect();torch.cuda.empty_cache()\n    return result\n\nresults=[]\nfor seed in (42,17,123):\n    order=(\'baseline\',\'attnres\') if seed==42 else (\'attnres\',\'baseline\')\n    for architecture in order:\n        results.append(train(architecture,seed))\n        dump_json(OUT/\'results.json\',results)\nprint(\'CONTROLLED_STAGE_COMPLETE\',flush=True)\n', 'phase2/backup.py': '"""Copy only this experiment\'s files to the explicitly authorized Drive folder."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\n\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for chunk in iter(lambda:handle.read(4*1024*1024),b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef verified_copy(source,destination):\n    source,destination=Path(source),Path(destination)\n    destination.parent.mkdir(parents=True,exist_ok=True)\n    expected=sha256(source)\n    if not destination.exists() or sha256(destination)!=expected:\n        temp=destination.with_name(destination.name+\'.copying\')\n        with source.open(\'rb\') as src,temp.open(\'wb\') as dst:\n            shutil.copyfileobj(src,dst,4*1024*1024)\n            dst.flush();os.fsync(dst.fileno())\n        assert sha256(temp)==expected,\'Backup verification failed\'\n        os.replace(temp,destination)\n    return {\'relative_destination\':str(destination),\'bytes\':source.stat().st_size,\'sha256\':expected}\n\n\ndef backup_tree(root,destination,include_checkpoints=True):\n    root,destination=Path(root),Path(destination)\n    records=[]\n    for path in sorted(root.rglob(\'*\')):\n        if not path.is_file() or any(p in (\'.venv\',\'.git\',\'__pycache__\') for p in path.relative_to(root).parts):\n            continue\n        if path.name.endswith((\'.tmp\',\'.copying\')) or path.name.endswith(\'_trace.json\'):\n            continue\n        if not include_checkpoints and path.suffix==\'.pt\':\n            continue\n        record=verified_copy(path,destination/path.relative_to(root))\n        records.append(record)\n        if path.suffix==\'.pt\':print(\'CHECKPOINT_BACKED_UP\',json.dumps(record),flush=True)\n    manifest=destination/\'backup_manifest.json\'\n    manifest.write_text(json.dumps(records,indent=2),encoding=\'utf-8\')\n    print(\'BACKUP_VERIFIED\',len(records),\'files\',sum(r[\'bytes\'] for r in records),\'bytes\',destination,flush=True)\n    return records\n', 'phase2/analyze_stage2.py': '"""Summarize measured phase-two artifacts without substituting missing runs."""\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport statistics as stats\n\nHERE=Path(__file__).resolve().parent\nROOT=HERE.parent\nOUT=HERE/\'analysis\'\nOUT.mkdir(exist_ok=True)\nruns=HERE/\'runs/controlled1000\'\nresults=[];configs=[];histories=[]\nfor seed in (17,42,123):\n    for architecture in (\'baseline\',\'attnres\'):\n        folder=runs/f\'{architecture}_seed{seed}\'\n        row=json.loads((folder/\'result.json\').read_text())\n        config=json.loads((folder/\'config.json\').read_text())\n        assert row[\'steps\']==1000 and row[\'training_tokens\']==4_096_000\n        assert row[\'architecture\']==architecture and row[\'seed\']==seed\n        assert config[\'residual_dtype\']==\'float32\'\n        results.append(row);configs.append(config)\n        for point in json.loads((folder/\'history.json\').read_text()):\n            histories.append({\'architecture\':architecture,\'seed\':seed,\'step\':point[\'step\'],**point[\'validation\']})\nassert len({json.dumps(c[\'source_hashes\'],sort_keys=True) for c in configs})==1\nassert len({c[\'data_manifest_sha256\'] for c in configs})==1\nassert len({c[\'holdout_manifest_sha256\'] for c in configs})==1\nfor name,expected in configs[0][\'source_hashes\'].items():\n    assert hashlib.sha256((HERE/name).read_bytes()).hexdigest()==expected,name\nassert len({r[\'engine\'] for r in results})==1\nassert len({json.dumps(r[\'environment\'],sort_keys=True) for r in results})==1\nsummary={}\nfor architecture in (\'baseline\',\'attnres\'):\n    group=[r for r in results if r[\'architecture\']==architecture]\n    fields={\n        \'validation_nll\':[r[\'validation\'][\'nll\'] for r in group],\n        \'validation_perplexity\':[r[\'validation\'][\'perplexity\'] for r in group],\n        \'heldout_nll\':[r[\'heldout\'][\'nll\'] for r in group],\n        \'heldout_perplexity\':[r[\'heldout\'][\'perplexity\'] for r in group],\n        \'training_tokens_per_second\':[r[\'median_training_tokens_per_second\'] for r in group],\n        \'first_step_seconds\':[r[\'first_step_seconds_including_cold_compile\'] for r in group],\n        \'run_wall_seconds\':[r[\'run_wall_seconds_including_compile_eval_checkpoint_io\'] for r in group],\n        \'peak_gpu_mib\':[r[\'peak_allocated_gpu_bytes_this_session\']/2**20 for r in group]}\n    summary[architecture]={\'parameters\':group[0][\'parameters\'],\n        **{key:{\'mean\':stats.mean(values),\'sample_sd\':stats.stdev(values)} for key,values in fields.items()}}\nby_key={(r[\'architecture\'],r[\'seed\']):r for r in results}\npaired=[]\nfor seed in (17,42,123):\n    a,b=by_key[\'baseline\',seed],by_key[\'attnres\',seed]\n    paired.append({\'seed\':seed,\n        \'heldout_nll_difference_attnres_minus_baseline\':b[\'heldout\'][\'nll\']-a[\'heldout\'][\'nll\'],\n        \'heldout_perplexity_reduction_percent\':100*(1-b[\'heldout\'][\'perplexity\']/a[\'heldout\'][\'perplexity\']),\n        \'validation_perplexity_reduction_percent\':100*(1-b[\'validation\'][\'perplexity\']/a[\'validation\'][\'perplexity\']),\n        \'throughput_change_percent\':100*(b[\'median_training_tokens_per_second\']/a[\'median_training_tokens_per_second\']-1)})\nbench=json.loads((HERE/\'results/benchmark_summary.json\').read_text())\nholdout=json.loads((HERE/\'data/holdout_manifest.json\').read_text())\nchecks=json.loads((HERE/\'results/controlled_compilation_checks.json\').read_text())\nbundle={\'summary\':summary,\'paired\':paired,\'results\':results,\'configs\':configs,\n    \'benchmark\':bench,\'holdout_manifest\':holdout,\'compilation_checks\':checks}\n(OUT/\'phase2_results.json\').write_text(json.dumps(bundle,indent=2),encoding=\'utf-8\')\nflat=[]\nfor r in results:\n    flat.append({k:v for k,v in r.items() if k not in (\'validation\',\'heldout\',\'environment\')}|\n        {f\'{prefix}_{k}\':v for prefix in (\'validation\',\'heldout\') for k,v in r[prefix].items()})\nfor name,rows in [(\'results.csv\',flat),(\'paired_comparisons.csv\',paired),(\'learning_curves.csv\',histories)]:\n    with (OUT/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)\n\ngain=stats.mean(r[\'heldout_perplexity_reduction_percent\'] for r in paired)\nspeed=stats.mean(r[\'throughput_change_percent\'] for r in paired)\nlines=[\'# mini-k3: precision-controlled training and speed optimization\',\'\',\n    \'Date: 18 October 2026. Author: Fazal Hameed.\',\'\',\n    \'## Main findings\',\'\',\n    f\'Three matched seeds completed 1,000 updates per architecture (4,096,000 predicted training tokens each). \'\n    f\'The mean paired change in heldout perplexity was a {gain:.2f}% reduction; \'\n    f\'the mean paired training-throughput change was {speed:+.2f}%. \'\n    \'These are measurements on a reduced dense MiniMind and one simple English dataset, not evidence of general reasoning or assistant quality.\',\'\',\n    \'Both architectures receive the same optimized training engine. A speedup over the original unoptimized implementation \'\n    \'does not establish a speedup over an equally optimized baseline. The raw-throughput and quality results must be read together.\',\'\',\n    (\'The modified model is still slower than the equally optimized baseline. The complete faster-and-stronger objective \'\n     \'has not yet been met on this comparison.\' if speed<0 else\n     \'The speed and quality claims apply only to this model, dataset, hardware and measured training path.\'),\'\',\n    \'## Research scope and implemented changes\',\'\',\n    \'The project adapts Block Attention Residuals from [Kimi K3, section 2.2, equations 8–10]\'\n    \'(https://arxiv.org/html/2607.24653v2#S2.SS2). It retains MiniMind grouped-query causal attention, rotary positions, \'\n    \'RMSNorm and its dense feed-forward network. Each attention/MLP is a sublayer. Four sublayers form a block. \'\n    \'A learned query scores RMS-normalized depth sources; a softmax weights the embedding, completed block sums and current partial sum. \'\n    \'A final depth mixer aggregates all completed blocks. No KDA, Stable LatentMoE, SiTU-GLU or Muon optimizer is implemented. \'\n    \'No KV cache or decoding-speed claim is included.\',\'\',\n    \'The original implementation repeatedly stacked and normalized old block outputs. The new implementation retains a stack \'\n    \'of completed block sums and reuses their inverse RMS values. Query and learned RMS weights combine algebraically in each score. \'\n    \'This retains gradients through the cached tensors; it is not detached activation caching. The singleton-source mixer returns \'\n    \'that source directly; its mathematically inactive query/norm parameters may have no gradient rather than an explicit zero.\',\'\',\n    \'The training path merges four batches of eight into one physical batch of 32, uses fused AdamW, and compiles the loss \'\n    \'with `torch.compile(fullgraph=True, dynamic=False, mode="default")` when both architectures pass the fixed speed rule. \'\n    \'Compilation is selected only if both controlled scaled-gradient checks pass, both benchmark candidates complete, \'\n    \'and each is at least 5% faster than its eager candidate. \'\n    \'The old meta-device rotary-position guard is omitted in this training-only wrapper because all model tensors are materialized.\',\'\',\n    \'## Precision issue found in the first pilot\',\'\',\n    \'The upstream attention residual uses an in-place addition whose FP16 destination rounds the running state under autocast. \'\n    \'The first AttnRes implementation promoted depth mixing to FP32 because its sources included the FP32 embedding, but its partial \'\n    \'block sums could still accumulate in FP16. Thus phase one had a precision difference as well as an architectural difference. \'\n    \'Its reported 23.9% perplexity reduction is a valid measurement of those implementations, but cannot be attributed solely to AttnRes.\',\'\',\n    \'This phase explicitly casts each attention/MLP output to FP32 before residual or partial-block accumulation in both architectures. \'\n    \'Model parameters and all residual/depth states remain FP32; linear operations use FP16 autocast. Both models are retrained from \'\n    \'the same shared initialization for each seed. The longer learning-rate schedule also differs from phase one, so cross-phase \'\n    \'quality differences do not isolate the numerical-precision effect. Historical phase-one source and results remain unchanged.\',\'\',\n    \'## Training and evaluation setup\',\'\',\n    \'- Free Tesla T4, 14.56 GiB; Python 3.13.15; PyTorch 2.11.0+cu130; Transformers 4.57.6.\',\n    \'- MiniMind pinned commit `f659b55761b754d306bd140573493a6543cafd7f`, Apache-2.0; license retained.\',\n    \'- Hidden width 384; six transformer layers; six query/two KV heads; vocabulary 6,400; input length 128.\',\n    \'- Baseline: 13,227,648 parameters. AttnRes: 13,237,632 parameters. Original MiniMind tokenizer.\',\n    \'- Seeds 42, 17, 123; baseline first at 42, AttnRes first at 17/123. Shared weights match exactly before training.\',\n    \'- 1,000 updates from scratch per run, physical/logical batch 32, 4,096 predicted tokens per update.\',\n    \'- AdamW fused; peak LR 3e-4; 50-step warmup and cosine decay toward 10% of peak; betas (0.9, 0.95); weight decay 0.1.\',\n    \'- FP16 linear autocast with FP32 accumulation; gradient scaling initially 1,024; gradient norm clipping at 1.0.\',\n    \'- Nonfinite loss/gradients stop a run rather than silently skipping updates. Same seed/step-dependent batches for both models.\',\n    \'- Training cache: 2,000,000 tokens. Blocks sampled with replacement; not a unique-token count or multiple full epochs.\',\n    \'- Validation: fixed 32,768 predicted tokens at step 0 and every 100 updates. Endpoint fixed at 1,000 updates.\',\n    f"- New heldout partition: {holdout[\'holdout_stories\']} accepted stories, 128,000 cached tokens; "\n    f"{results[0][\'heldout\'][\'tokens\']:,} predicted tokens in full packed blocks, evaluated only at the fixed endpoint.",\'\',\n    \'Data uses [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories), pinned revision \'\n    \'`f54c09fd23315a6f9c86f9dc80f725de7d8f9c64`, CDLA-Sharing-1.0. EOS is inserted between stories. \'\n    \'Train and original validation token caches are reconstructed and checked for exact equality before creating the new partition. \'\n    \'The holdout starts after the last complete story contributing to the original validation prefix. Exact text hashes exclude \'\n    \'selected train stories, original validation stories and repeated holdout stories. This is an extra partition of the official \'\n    \'validation split, not an official test split or an independent domain. Near-duplicate/paraphrase removal was not performed. \'\n    \'The holdout was not used to tune the learning rate, architecture, budget or engine.\',\'\',\n    \'Evaluation reports mean next-token negative log-likelihood in nats per predicted token and perplexity exp(NLL). \'\n    \'Packed blocks can cross story boundaries and do not mask attention at EOS. The same packing and causal mask apply to both models.\',\'\',\n    \'## Per-seed measured results\',\'\',\n    \'| Seed | Model | Validation NLL | Heldout NLL | Heldout perplexity | Training tokens/s | First step, s | Full run, s |\',\n    \'| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |\']\nfor r in results:\n    lines.append(f"| {r[\'seed\']} | {r[\'architecture\']} | {r[\'validation\'][\'nll\']:.4f} | {r[\'heldout\'][\'nll\']:.4f} | "\n        f"{r[\'heldout\'][\'perplexity\']:.3f} | {r[\'median_training_tokens_per_second\']:.0f} | "\n        f"{r[\'first_step_seconds_including_cold_compile\']:.2f} | {r[\'run_wall_seconds_including_compile_eval_checkpoint_io\']:.2f} |")\nlines+=[\'\',\'Mean ± sample standard deviation across three seeds:\',\'\',\n    \'| Model | Heldout NLL | Heldout perplexity | Training tokens/s | Peak GPU MiB |\',\n    \'| --- | ---: | ---: | ---: | ---: |\']\nfor architecture,s in summary.items():\n    def fmt(key,places):return f"{s[key][\'mean\']:.{places}f} ± {s[key][\'sample_sd\']:.{places}f}"\n    lines.append(f"| {architecture} | {fmt(\'heldout_nll\',4)} | {fmt(\'heldout_perplexity\',3)} | {fmt(\'training_tokens_per_second\',0)} | {fmt(\'peak_gpu_mib\',1)} |")\nlines+=[\'\',\'## Throughput trials before longer training\',\'\',\n    \'Each isolated trial used eight warmup and 60 measured updates at a logical batch of 32. \'\n    \'Trials started from scratch and are performance experiments, not heldout-quality results. \'\n    \'The cached AttnRes benchmark preserves phase-one FP16 partial-block accumulation; the subsequent controlled trainer \'\n    \'uses FP32 for those sums. Its actual training speeds are therefore reported separately above.\',\'\',\n    \'| Candidate | Median tokens/s | First step including compilation, s | Peak GPU MiB |\',\n    \'| --- | ---: | ---: | ---: |\']\nfor row in bench:\n    if row[\'status\']==\'ok\':\n        lines.append(f"| {row[\'config\'][\'name\']} | {row[\'median_training_tokens_per_second\']:.0f} | "\n            f"{row[\'first_step_seconds_including_cold_compile\']:.2f} | {row[\'peak_allocated_gpu_bytes\']/2**20:.1f} |")\n    else:lines.append(f"| {row[\'config\'][\'name\']} | {row[\'status\']} | — | — |")\nlines+=[\'\',\n    \'Timing uses synchronized complete optimizer updates, including batch construction/transfer, forward/backward, gradient scaling, \'\n    \'clipping and optimizer work. Steady throughput is the median of per-step tokens/s after the first eight steps. \'\n    \'It excludes validation, compilation setup, checkpoint writes, Drive backup and data/dependency setup. \'\n    \'The first-step measurement includes any compilation on that invocation; later runs can reuse compiler caches, so \'\n    \'it is not a guaranteed cold-cache compilation cost for every seed. Full-run wall time includes model setup, compilation, \'\n    \'evaluation and periodic checkpoint/Drive backup IO, but excludes data preparation, imports, and final result-metadata backup. \'\n    \'GPU memory is PyTorch peak allocated memory during each uninterrupted run, not reserved memory or full-device use.\',\'\',\n    \'The benchmark loop also reduces per-microbatch Python checks relative to the historical pilot. \'\n    \'Consequently its eager original timings differ from phase-one timings; one must not attribute the entire \'\n    \'cross-phase throughput gain to depth-mixing optimization. Free shared hardware, compiler-cache reuse, \'\n    \'single-session timing and limited execution-order reversal restrict performance claims.\',\'\',\n    \'## Verification, artifacts and reproducibility\',\'\',\n    \'Four optimized-wrapper checks passed in Colab: original-path equality, cached/reference logits and gradients \'\n    \'(including nonzero learned queries and multiple block sizes), causal outputs, and CUDA FP16 equivalence. \'\n    \'The controlled-model checks verify exact shared initialization, FP32 reference outputs/gradients and causality, \'\n    \'and FP32 states entering every layer normalization under CUDA FP16 autocast. \'\n    \'Compiled and eager losses and gradients were compared at the real batch/context shape for both controlled architectures, \'\n    \'using loss scaling of 1,024 before backward and unscaling gradients afterward, as in training. \'\n    \'Tolerances and maximum observed errors are saved in `controlled_compilation_checks.json`; tests are not quality results.\',\'\',\n    \'An earlier unscaled FP16 backward comparison failed the gradient tolerance despite nearly identical losses. \'\n    \'Diagnostics were retained under `unscaled_*` in `phase2/results/`. Repeating the test with the trainer’s loss scaling \'\n    \'greatly reduced differences and both models passed the same tolerance. This is consistent with FP16 small-gradient \'\n    \'underflow/rounding in the unscaled computation; it does not establish bitwise equivalence or prove compiler correctness \'\n    \'for every trained state. No tolerance was loosened to obtain the scaled result.\',\'\',\n    \'Every run stores its configuration, source/data hashes, per-step timings, validation history, endpoint result and \'\n    \'a resumable checkpoint containing model, optimizer, scaler, RNG state and step. Runtime checkpoints are saved every \'\n    \'100 updates; verified Drive copies are made at 500 and 1,000 updates and after completion. \'\n    \'Changing the source/configuration/data/environment requires a new run directory. The original phase-one trainer \'\n    \'also passed interrupted-resume equality tests; the phase-two runner reuses the same checkpoint-state pattern but \'\n    \'has not undergone a full GPU interrupted/resumed trajectory equivalence test.\',\'\',\n    \'User-authorized Drive destination: `My Drive/mini-k3/2026-10-18`. Each copied file is verified by SHA-256. \'\n    \'Phase-one and phase-two model bytes live there; the local research bundle contains code, logs, manifests and reports, \'\n    \'and does not pretend to contain model checkpoint bytes. A complete backup manifest records sizes and checksums.\',\'\',\n    \'## Interpretation and remaining work\',\'\',\n    \'This study measures early domain language modeling on a 13.2M-parameter model. Three seeds quantify \'\n    \'initialization/sampling variability on a fixed corpus, not statistical confidence across tasks or domains. \'\n    \'The unchanged hyperparameters may favor one architecture; no learning-rate sweep was conducted. \'\n    \'A longer or full-size MiniMind study, independent-domain evaluation, downstream tasks, cached inference and \'\n    \'wall-clock-budget comparisons remain. Results should be described with these limits; a professor’s acceptance \'\n    \'decision is independent of completing this experiment.\',\'\',\n    \'\']\n(OUT/\'PHASE2_REPORT.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfig,axes=plt.subplots(1,2,figsize=(11,4))\nfor architecture,color in [(\'baseline\',\'#3659a5\'),(\'attnres\',\'#d46423\')]:\n    points=sorted({r[\'step\'] for r in histories})\n    values=[[r[\'nll\'] for r in histories if r[\'architecture\']==architecture and r[\'step\']==step] for step in points]\n    means=[stats.mean(v) for v in values];sd=[stats.stdev(v) for v in values]\n    xs=[s*4096/1e6 for s in points]\n    axes[0].plot(xs,means,label=architecture,color=color,marker=\'.\',markersize=5)\n    axes[0].fill_between(xs,[m-s for m,s in zip(means,sd)],[m+s for m,s in zip(means,sd)],alpha=.15,color=color)\n    ys=[by_key[architecture,seed][\'heldout\'][\'perplexity\'] for seed in (17,42,123)]\n    axes[1].scatter([architecture]*3,ys,color=color,s=35,label=architecture)\naxes[0].set(xlabel=\'Predicted training tokens (millions)\',ylabel=\'Validation NLL (nats/token)\',title=\'Mean ± sample SD, three seeds\')\naxes[0].legend();axes[0].grid(alpha=.2)\naxes[1].set(ylabel=\'Heldout perplexity (lower is better)\',title=\'Untouched stories: fixed 1,000-step endpoint\')\naxes[1].grid(axis=\'y\',alpha=.2)\nfig.tight_layout();fig.savefig(OUT/\'phase2_results.png\',dpi=180);plt.close(fig)\nprint(json.dumps({\'summary\':summary,\'paired\':paired},indent=2))\nprint(\'PHASE2_ANALYSIS_COMPLETE\',OUT/\'PHASE2_REPORT.md\')\n', 'phase2/profile_stage2.py': '"""Profile the original implementations without changing phase-one checkpoints."""\nimport gc\nimport json\nfrom pathlib import Path\nimport torch\nfrom torch.profiler import profile, ProfilerActivity, record_function\nfrom experiment import build_model, token_blocks, training_batch, next_token_loss, environment\n\nROOT = Path(__file__).resolve().parent.parent\nOUT = ROOT / \'phase2\' / \'results\'\nOUT.mkdir(parents=True, exist_ok=True)\nassert torch.cuda.is_available(), \'Reconnect to a T4 GPU runtime.\'\ntorch.set_num_threads(4)\ndata = token_blocks(ROOT / \'data/train.npy\', 128)\nsummaries = {}\nfor variant in (\'baseline\', \'attnres\'):\n    model = build_model(\'pilot\', 128, variant, 42).cuda().train()\n    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(.9,.95), weight_decay=.1)\n    scaler = torch.amp.GradScaler(\'cuda\', init_scale=1024)\n    def step(index):\n        optimizer.zero_grad(set_to_none=True)\n        for micro in range(4):\n            x,y = training_batch(data,42,index,micro,8,\'cuda\')\n            with torch.autocast(\'cuda\',dtype=torch.float16):\n                loss = next_token_loss(model,x,y) / 4\n            scaler.scale(loss).backward()\n        scaler.unscale_(optimizer)\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)\n        scaler.step(optimizer)\n        scaler.update()\n    for i in range(3):\n        step(i)\n    torch.cuda.synchronize()\n    with profile(activities=[ProfilerActivity.CPU,ProfilerActivity.CUDA],\n                 record_shapes=True, profile_memory=True) as prof:\n        with record_function(variant):\n            for i in range(3,5):\n                step(i)\n        torch.cuda.synchronize()\n    events=[]\n    for e in prof.key_averages():\n        events.append({\'name\':e.key,\'calls\':e.count,\n            \'self_cpu_us\':float(e.self_cpu_time_total),\n            \'self_device_us\':float(getattr(e,\'self_device_time_total\',0)),\n            \'device_total_us\':float(getattr(e,\'device_time_total\',0))})\n    events.sort(key=lambda e:e[\'self_device_us\'],reverse=True)\n    summaries[variant]={\'environment\':environment(\'cuda\'),\'profiled_steps\':2,\'events\':events}\n    print(\'\\nPROFILE\',variant,flush=True)\n    print(json.dumps(events[:20],indent=2),flush=True)\n    prof.export_chrome_trace(str(OUT/f\'{variant}_trace.json\'))\n    del model,optimizer,scaler,prof\n    gc.collect()\n    torch.cuda.empty_cache()\n(OUT/\'original_profiles.json\').write_text(json.dumps(summaries,indent=2))\nprint(\'PROFILE_COMPLETE\',OUT/\'original_profiles.json\',flush=True)\n', 'phase2/export_logs.py': '"""Export small measured artifacts through visible Colab output, with integrity checks."""\nimport base64\nimport hashlib\nimport io\nfrom pathlib import Path\nimport zipfile\n\nROOT=Path(__file__).resolve().parent.parent\nbuffer=io.BytesIO()\nwith zipfile.ZipFile(buffer,\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n    for path in sorted((ROOT/\'phase2\').rglob(\'*\')):\n        if not path.is_file() or \'__pycache__\' in path.parts or path.name.endswith(\'_trace.json\'):\n            continue\n        if path.suffix in (\'.json\',\'.csv\',\'.md\') or path.name==\'test_controlled_unscaled.py\':\n            archive.write(path,path.relative_to(ROOT))\n    for name in (\'drive_backup_final.json\',\'drive_backup_verification.json\'):\n        if (ROOT/name).exists():archive.write(ROOT/name,name)\nblob=buffer.getvalue()\nencoded=base64.b64encode(blob).decode(\'ascii\')\nchunks=[encoded[i:i+6000] for i in range(0,len(encoded),6000)]\nprint(\'P2LOG_SIZE:\'+str(len(blob)))\nprint(\'P2LOG_SHA256:\'+hashlib.sha256(blob).hexdigest())\nprint(\'P2LOG_CHUNK_COUNT:\'+str(len(chunks)))\nfor index,chunk in enumerate(chunks):print(f\'P2LOG_CHUNK_{index}:\'+chunk)\n'}
for name,source in PAYLOAD.items():
    target=WORK/name
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_bytes(source.encode('utf-8'))
def run(command):
    with subprocess.Popen(command,cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
        for line in p.stdout:print(line,end='',flush=True)
        status=p.wait()
    if status:raise RuntimeError(f'Experiment command failed: {status}')
print('Pinned source ready',WORK)


In [ ]:
PHASE3_SOURCES={'models.py': '"""Tuple-based Block AttnRes; all residual and block accumulation is FP32."""\nimport torch\nfrom torch import nn\nfrom experiment import make_config\nfrom model.model_minimind import MiniMindForCausalLM\nfrom attnres import BlockAttnResModel\nfrom optimized import TrainingLM\n\nCONFIGS={\n    \'baseline6\':{\'architecture\':\'baseline\',\'layers\':6,\'block_sublayers\':4},\n    \'baseline5\':{\'architecture\':\'baseline\',\'layers\':5,\'block_sublayers\':4},\n    \'attnres6_b4\':{\'architecture\':\'attnres\',\'layers\':6,\'block_sublayers\':4},\n    \'attnres6_b12\':{\'architecture\':\'attnres\',\'layers\':6,\'block_sublayers\':12},\n    \'attnres5_b4\':{\'architecture\':\'attnres\',\'layers\':5,\'block_sublayers\':4},\n    \'attnres5_b12\':{\'architecture\':\'attnres\',\'layers\':5,\'block_sublayers\':12},\n}\n\n\nclass TupleLM(TrainingLM):\n    @staticmethod\n    def mix_tuple(mixer,sources,inverses):\n        if len(sources)==1:return sources[0]\n        query=mixer.query.float()*mixer.key_norm.weight.float()\n        scores=[(value*query).sum(-1)*inverse.squeeze(-1) for value,inverse in zip(sources,inverses)]\n        if len(sources)==2:\n            # Exact two-way softmax identity, with FP32 roundoff differences.\n            gate=(scores[1]-scores[0]).sigmoid().unsqueeze(-1)\n            return sources[0]+gate*(sources[1]-sources[0])\n        weights=torch.stack(scores,0).softmax(0)\n        mixed=sources[0]*weights[0].unsqueeze(-1)\n        for index in range(1,len(sources)):\n            mixed=mixed+sources[index]*weights[index].unsqueeze(-1)\n        return mixed\n\n    def forward(self,x):\n        if self.mode==\'fp32\':return super().forward(x)\n        body=self.backbone\n        embedding=body.dropout(body.embed_tokens(x)).float()\n        positions=(body.freqs_cos[:x.shape[1]],body.freqs_sin[:x.shape[1]])\n        completed=(embedding,)\n        inverses=(self.inverse_rms(embedding,body.config.rms_norm_eps),)\n        partial,index=None,0\n        for layer in body.layers:\n            for attention in (True,False):\n                sources=completed if partial is None else completed+(partial,)\n                rms=inverses if partial is None else inverses+(self.inverse_rms(partial,body.config.rms_norm_eps),)\n                hidden=self.mix_tuple(body.depth_mixers[index],sources,rms)\n                if attention:\n                    delta,_=layer.self_attn(layer.input_layernorm(hidden),positions,use_cache=False)\n                else:delta=layer.mlp(layer.post_attention_layernorm(hidden))\n                delta=delta.float()\n                partial=delta if partial is None else partial+delta\n                index+=1\n                if index%body.block_sublayers==0:\n                    completed=completed+(partial,)\n                    inverses=inverses+(self.inverse_rms(partial,body.config.rms_norm_eps),)\n                    partial=None\n        sources=completed if partial is None else completed+(partial,)\n        rms=inverses if partial is None else inverses+(self.inverse_rms(partial,body.config.rms_norm_eps),)\n        hidden=self.mix_tuple(body.depth_mixers[index],sources,rms)\n        return self.lm_head(body.norm(hidden))\n\n\ndef original_model(name,seed=42,size=\'pilot\'):\n    spec=CONFIGS[name]\n    config=make_config(size,128)\n    config.num_hidden_layers=spec[\'layers\'] if size==\'pilot\' else 2\n    torch.manual_seed(seed)\n    original=MiniMindForCausalLM(config)\n    if spec[\'architecture\']==\'attnres\':\n        body=BlockAttnResModel(config,spec[\'block_sublayers\'])\n        missing,unexpected=body.load_state_dict(original.model.state_dict(),strict=False)\n        assert not unexpected and all(k.startswith(\'depth_mixers.\') for k in missing)\n        original.model=body\n        if config.tie_word_embeddings:original.model.embed_tokens.weight=original.lm_head.weight\n    return original\n\n\ndef create_model(name,seed=42,size=\'pilot\'):\n    original=original_model(name,seed,size)\n    return TupleLM(original,\'fp32\' if CONFIGS[name][\'architecture\']==\'baseline\' else \'attnres_cached\')\n', 'test_models.py': '"""Math, causality, shared initialization, FP32 states, and compiled gradients."""\nimport copy\nimport gc\nimport json\nfrom pathlib import Path\nimport unittest\nimport torch\nfrom models import CONFIGS,TupleLM,create_model,original_model\nfrom controlled import ControlledLM\nfrom optimized import TrainingLoss\n\n\nclass ModelTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):torch.set_num_threads(2)\n\n    def test_shared_initialization(self):\n        for depth in (5,6):\n            baseline=create_model(f\'baseline{depth}\').state_dict()\n            attnres=create_model(f\'attnres{depth}_b12\').state_dict()\n            for name,tensor in baseline.items():torch.testing.assert_close(tensor,attnres[name],rtol=0,atol=0)\n\n    def test_tuple_matches_reference_and_is_causal(self):\n        for name in (\'attnres6_b4\',\'attnres6_b12\',\'attnres5_b4\',\'attnres5_b12\'):\n            original=original_model(name,size=\'tiny\')\n            with torch.no_grad():\n                for mixer in original.model.depth_mixers:\n                    mixer.query.normal_(0,.1);mixer.key_norm.weight.uniform_(.8,1.2)\n            a=ControlledLM(copy.deepcopy(original),\'attnres\')\n            b=TupleLM(copy.deepcopy(original),\'attnres_cached\')\n            x=torch.randint(0,6400,(2,12));y=x.clone();y[:,7:]=0\n            torch.testing.assert_close(a(x),b(x),rtol=3e-4,atol=3e-6)\n            torch.testing.assert_close(b(x)[:,:7],b(y)[:,:7],rtol=1e-5,atol=1e-5)\n            TrainingLoss(a)(x,y).backward();TrainingLoss(b)(x,y).backward()\n            for (pname,p),(_,q) in zip(a.named_parameters(),b.named_parameters()):\n                gp=torch.zeros_like(p) if p.grad is None else p.grad\n                gq=torch.zeros_like(q) if q.grad is None else q.grad\n                torch.testing.assert_close(gp,gq,rtol=.003,atol=4e-6,msg=pname)\n\n    @unittest.skipUnless(torch.cuda.is_available(),\'CUDA runs in Colab\')\n    def test_fp32_residual_states(self):\n        for name in CONFIGS:\n            model=create_model(name).cuda();seen=[]\n            hooks=[norm.register_forward_pre_hook(lambda m,args:seen.append(args[0].dtype))\n                for layer in model.backbone.layers for norm in (layer.input_layernorm,layer.post_attention_layernorm)]\n            with torch.autocast(\'cuda\',dtype=torch.float16):model(torch.ones((2,128),dtype=torch.long,device=\'cuda\'))\n            self.assertTrue(seen and all(dtype==torch.float32 for dtype in seen))\n            for hook in hooks:hook.remove()\n            del model;gc.collect();torch.cuda.empty_cache()\n\n\ndef gpu_compilation_checks():\n    rows=[]\n    for name in CONFIGS:\n        a=TrainingLoss(create_model(name).cuda());b=copy.deepcopy(a)\n        compiled=torch.compile(b,fullgraph=True,dynamic=False,mode=\'default\')\n        generator=torch.Generator(device=\'cuda\').manual_seed(19)\n        x=torch.randint(0,6400,(32,128),generator=generator,device=\'cuda\')\n        y=torch.randint(0,6400,(32,128),generator=generator,device=\'cuda\')\n        with torch.autocast(\'cuda\',dtype=torch.float16):la=a(x,y);lb=compiled(x,y)\n        torch.testing.assert_close(la,lb,rtol=.001,atol=.001)\n        (la*1024).backward();(lb*1024).backward()\n        maximum=0.\n        for (key,p),(_,q) in zip(a.named_parameters(),b.named_parameters()):\n            gp=torch.zeros_like(p) if p.grad is None else p.grad/1024\n            gq=torch.zeros_like(q) if q.grad is None else q.grad/1024\n            maximum=max(maximum,(gp-gq).abs().max().item())\n            torch.testing.assert_close(gp,gq,rtol=.04,atol=2e-4,msg=key)\n        row={\'candidate\':name,\'status\':\'passed\',\'loss_absolute_error\':abs(la.item()-lb.item()),\n            \'max_gradient_absolute_error\':maximum,\'loss_scale\':1024,\'gradient_rtol\':.04,\'gradient_atol\':2e-4}\n        rows.append(row);print(\'PHASE3_COMPILE_CHECK\',json.dumps(row),flush=True)\n        del a,b,compiled,la,lb;gc.collect();torch.cuda.empty_cache()\n    path=Path(__file__).parent/\'results/compilation_checks.json\'\n    path.parent.mkdir(parents=True,exist_ok=True);path.write_text(json.dumps(rows,indent=2))\n\n\nif __name__==\'__main__\':\n    outcome=unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(ModelTests))\n    if not outcome.wasSuccessful():raise SystemExit(1)\n    if torch.cuda.is_available():gpu_compilation_checks()\n', 'prepare_holdout.py': '"""Reconstruct preceding caches before reserving subsequent complete stories."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom datasets import load_dataset\nfrom transformers import AutoTokenizer\nfrom experiment import DATASET,DATA_REVISION,digest,dump_json\n\nROOT=Path(__file__).resolve().parent.parent\nOUT=ROOT/\'phase3/data\'\nOUT.mkdir(parents=True,exist_ok=True)\nif (OUT/\'holdout_manifest.json\').exists():\n    manifest=json.loads((OUT/\'holdout_manifest.json\').read_text())\n    assert digest(OUT/\'holdout.npy\')==manifest[\'sha256\']\n    print(\'PHASE3_HOLDOUT_VERIFIED\',json.dumps(manifest),flush=True)\n    raise SystemExit(0)\ntokenizer=AutoTokenizer.from_pretrained(str(ROOT/\'vendor/model\'),trust_remote_code=False)\nseen=set()\ndef encode(row):\n    text=row[\'text\'].strip()\n    if not text:return None\n    key=hashlib.sha256(text.encode(\'utf-8\')).digest()\n    if key in seen:return None\n    seen.add(key)\n    return tokenizer.encode(text,add_special_tokens=False)+[tokenizer.eos_token_id]\ntrain=[];train_count=0\nfor row in load_dataset(DATASET,revision=DATA_REVISION,split=\'train\',streaming=True):\n    tokens=encode(row)\n    if tokens is None:continue\n    train.extend(tokens);train_count+=1\n    if len(train)>=2_000_000:break\nassert np.array_equal(np.asarray(train[:2_000_000],dtype=np.int32),np.load(ROOT/\'data/train.npy\'))\nprint(\'PHASE3_TRAIN_PREFIX_VERIFIED\',train_count,flush=True)\ngroups=[[],[],[]];counts=[0,0,0];caps=[100_000,128_000,128_000]\nindex=0\nfor row in load_dataset(DATASET,revision=DATA_REVISION,split=\'validation\',streaming=True):\n    tokens=encode(row)\n    if tokens is None:continue\n    groups[index].extend(tokens);counts[index]+=1\n    if len(groups[index])>=caps[index]:\n        index+=1\n        if index==3:break\nassert index==3\nassert np.array_equal(np.asarray(groups[0][:100_000],dtype=np.int32),np.load(ROOT/\'data/validation.npy\'))\nassert np.array_equal(np.asarray(groups[1][:128_000],dtype=np.int32),np.load(ROOT/\'phase2/data/holdout.npy\'))\nnp.save(OUT/\'holdout.npy\',np.asarray(groups[2][:128_000],dtype=np.int32))\nmanifest={\'dataset\':DATASET,\'revision\':DATA_REVISION,\'tokens\':128_000,\n    \'sha256\':digest(OUT/\'holdout.npy\'),\'train_stories\':train_count,\n    \'original_validation_stories\':counts[0],\'phase2_holdout_stories\':counts[1],\'new_holdout_stories\':counts[2],\n    \'selection\':\'Subsequent complete validation stories, exact-text disjoint from all previously selected complete train/validation/heldout stories\',\n    \'previous_data_sha256\':digest(ROOT/\'data/data_manifest.json\'),\n    \'phase2_manifest_sha256\':digest(ROOT/\'phase2/data/holdout_manifest.json\'),\n    \'role\':\'Endpoint confirmation only; not an official test split or independent domain.\'}\ndump_json(OUT/\'holdout_manifest.json\',manifest)\nprint(\'PHASE3_HOLDOUT_READY\',json.dumps(manifest),flush=True)\n', 'runner.py': '"""Fixed-budget development screen and locked three-seed confirmation."""\nimport argparse\nimport gc\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport statistics\nimport time\nimport torch\nfrom experiment import digest,dump_json,environment,token_blocks\nfrom optimized import TrainingLoss\nfrom benchmark_stage2 import logical_batch\nfrom backup import backup_tree,verified_copy\nfrom models import CONFIGS,create_model\n\nROOT=Path(__file__).resolve().parent.parent\nHERE=ROOT/\'phase3\'\nTARGET=2.10\ntorch.set_num_threads(4)\ntorch.backends.cuda.matmul.allow_tf32=False\ntorch.backends.cudnn.allow_tf32=False\nassert torch.cuda.is_available()\nchecks=json.loads((HERE/\'results/compilation_checks.json\').read_text())\nassert {r[\'candidate\'] for r in checks if r[\'status\']==\'passed\'}==set(CONFIGS),\'All compiled model checks must pass\'\ndata=token_blocks(ROOT/\'data/train.npy\',128)\nvalidation=token_blocks(ROOT/\'data/validation.npy\',128)[:256]\nsource_hashes={name:digest(HERE/name) for name in [\'models.py\',\'runner.py\',\'test_models.py\',\'prepare_holdout.py\',\'PROTOCOL.md\',\n    \'../phase2/optimized.py\',\'../phase2/benchmark_stage2.py\',\'../phase2/controlled.py\',\'../attnres.py\',\'../experiment.py\']}\nDRIVE=Path(os.environ[\'MINIK3_DRIVE_BACKUP\']) if os.environ.get(\'MINIK3_DRIVE_BACKUP\') else None\n\n\n@torch.no_grad()\ndef evaluate(model,blocks):\n    model.eval();total,tokens=0.,0\n    loss_fn=TrainingLoss(model)\n    for lo in range(0,len(blocks),32):\n        pack=blocks[lo:lo+32].cuda()\n        with torch.autocast(\'cuda\',dtype=torch.float16):loss=loss_fn(pack[:,:-1],pack[:,1:])\n        count=pack.shape[0]*128\n        total+=loss.item()*count;tokens+=count\n    model.train()\n    return {\'nll\':total/tokens,\'perplexity\':math.exp(total/tokens),\'tokens\':tokens}\n\n\ndef train(name,seed,phase):\n    wall_start=time.perf_counter();previous_wall=0.\n    folder=HERE/\'runs\'/phase/f\'{name}_seed{seed}\'\n    folder.mkdir(parents=True,exist_ok=True)\n    spec={\'candidate\':name,**CONFIGS[name],\'phase\':phase,\'seed\':seed,\'steps\':1000,\'sequence\':128,\n        \'physical_batch\':32,\'logical_batch\':32,\'lr\':3e-4,\'warmup_steps\':50,\'min_lr_fraction\':.1,\n        \'optimizer\':\'AdamW fused\',\'betas\':[.9,.95],\'weight_decay\':.1,\'gradient_clip\':1.,\n        \'initial_loss_scale\':1024,\'residual_dtype\':\'float32\',\'linear_autocast\':\'float16\',\n        \'engine\':\'inductor fullgraph default dynamicFalse\',\'target_development_nll\':TARGET,\'evaluate_every\':50,\n        \'source_hashes\':source_hashes,\'data_manifest_sha256\':digest(ROOT/\'data/data_manifest.json\'),\n        \'compilation_checks_sha256\':digest(HERE/\'results/compilation_checks.json\'),\n        \'holdout_manifest_sha256\':digest(HERE/\'data/holdout_manifest.json\'),\n        \'selection_sha256\':digest(HERE/\'selection.json\') if phase==\'confirmation\' else None,\n        \'environment\':environment(\'cuda\')}\n    spec=json.loads(json.dumps(spec))\n    if (folder/\'config.json\').exists():\n        assert json.loads((folder/\'config.json\').read_text())==spec,\'Changed source/configuration: use a new output directory\'\n    dump_json(folder/\'config.json\',spec)\n    if (folder/\'result.json\').exists():\n        print(\'PHASE3_ALREADY_COMPLETE\',name,seed,phase,flush=True)\n        return json.loads((folder/\'result.json\').read_text())\n    model=create_model(name,seed).cuda().train()\n    objective=TrainingLoss(model)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,betas=(.9,.95),weight_decay=.1,fused=True)\n    scaler=torch.amp.GradScaler(\'cuda\',init_scale=1024)\n    history,timings,start=[],[],0\n    checkpoint=folder/\'latest.pt\'\n    if checkpoint.exists():\n        saved=torch.load(checkpoint,map_location=\'cpu\',weights_only=True)\n        assert saved[\'spec\']==spec\n        model.load_state_dict(saved[\'model\']);optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\'])\n        history=saved[\'history\'];timings=saved[\'timings\'];start=saved[\'step\'];previous_wall=saved[\'elapsed_wall\']\n        torch.set_rng_state(saved[\'cpu_rng\']);torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        print(\'PHASE3_RESUME\',name,seed,start,flush=True)\n    else:\n        metrics=evaluate(model,validation)\n        history.append({\'step\':0,**metrics,\'training_seconds\':0.,\'wall_seconds\':time.perf_counter()-wall_start})\n    objective=torch.compile(objective,fullgraph=True,dynamic=False,mode=\'default\')\n    torch.cuda.reset_peak_memory_stats()\n    cumulative_training=sum(r[\'seconds\'] for r in timings)\n    def save(step):\n        state={\'spec\':spec,\'model\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\n            \'step\':step,\'history\':history,\'timings\':timings,\'elapsed_wall\':previous_wall+time.perf_counter()-wall_start,\n            \'cpu_rng\':torch.get_rng_state(),\'cuda_rng\':torch.cuda.get_rng_state_all()}\n        temporary=checkpoint.with_suffix(\'.pt.tmp\')\n        torch.save(state,temporary);os.replace(temporary,checkpoint)\n        dump_json(folder/\'history.json\',history);dump_json(folder/\'timing.json\',timings)\n    for step in range(start,1000):\n        lr=3e-4*min(1.,(step+1)/50)*(.1+.9*(1+math.cos(math.pi*max(0,step-50)/950))/2)\n        for group in optimizer.param_groups:group[\'lr\']=lr\n        optimizer.zero_grad(set_to_none=True)\n        torch.cuda.synchronize();began=time.perf_counter()\n        x,y=logical_batch(data,seed,step)\n        with torch.autocast(\'cuda\',dtype=torch.float16):loss=objective(x,y)\n        scaler.scale(loss).backward();scaler.unscale_(optimizer)\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True)\n        if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite loss; paired run stopped\')\n        scaler.step(optimizer);scaler.update()\n        torch.cuda.synchronize();elapsed=time.perf_counter()-began\n        cumulative_training+=elapsed\n        timings.append({\'step\':step+1,\'seconds\':elapsed,\'tokens\':4096})\n        if (step+1)%100==0 or step==start:\n            print(\'PHASE3_TRAIN\',phase,name,seed,step+1,\'nll\',round(loss.item(),4),\'tokens/s\',round(4096/elapsed),flush=True)\n        if (step+1)%50==0:\n            metrics=evaluate(model,validation)\n            row={\'step\':step+1,**metrics,\'training_seconds\':cumulative_training,\n                \'wall_seconds\':previous_wall+time.perf_counter()-wall_start}\n            history.append(row)\n            if (step+1)%100==0:print(\'PHASE3_VALIDATION\',phase,name,seed,json.dumps(row),flush=True)\n        if (step+1)%250==0:save(step+1)\n    heldout=None\n    if phase==\'confirmation\':heldout=evaluate(model,token_blocks(HERE/\'data/holdout.npy\',128))\n    crossing=next((r for r in history if r[\'nll\']<=TARGET),None)\n    steady=[r for r in timings if r[\'step\']>8]\n    result={\'candidate\':name,\'phase\':phase,\'seed\':seed,\'steps\':1000,\'training_tokens\':4_096_000,\n        \'parameters\':sum(p.numel() for p in model.parameters()),\'development\':history[-1],\'heldout\':heldout,\n        \'target_nll\':TARGET,\'first_observed_target_crossing\':crossing,\n        \'median_training_tokens_per_second\':statistics.median(4096/r[\'seconds\'] for r in steady),\n        \'first_step_seconds_including_compilation\':timings[0][\'seconds\'],\'training_step_seconds\':cumulative_training,\n        \'active_run_wall_seconds_including_setup_compile_eval_checkpoint_io\':previous_wall+time.perf_counter()-wall_start,\n        \'peak_allocated_gpu_bytes_this_session\':torch.cuda.max_memory_allocated(),\'resume_step_this_session\':start,\n        \'persistent_checkpoint_contents\':\'model weights and config; optimizer/scaler only in temporary runtime latest.pt\',\n        \'environment\':spec[\'environment\']}\n    dump_json(folder/\'result.json\',result)\n    # Completed model-only export conserves Drive storage and cannot resume AdamW exactly.\n    weights=folder/\'model_weights.pt\'\n    torch.save({\'model\':model.state_dict(),\'spec\':spec,\'result\':result},weights)\n    if DRIVE:\n        records=backup_tree(folder,DRIVE/folder.relative_to(ROOT),include_checkpoints=False)\n        records.append(verified_copy(weights,DRIVE/weights.relative_to(ROOT)))\n        dump_json(folder/\'backup_verification.json\',records)\n        verified_copy(folder/\'backup_verification.json\',DRIVE/folder.relative_to(ROOT)/\'backup_verification.json\')\n        print(\'PHASE3_MODEL_BACKUP_VERIFIED\',name,seed,weights.stat().st_size,flush=True)\n    print(\'PHASE3_RESULT\',json.dumps(result),flush=True)\n    del model,objective,optimizer,scaler\n    gc.collect();torch.cuda.empty_cache()\n    return result\n\n\ndef screen():\n    results=[]\n    for name in CONFIGS:\n        results.append(train(name,42,\'screen\'))\n        dump_json(HERE/\'screen_results.json\',results)\n    baseline=next(r for r in results if r[\'candidate\']==\'baseline6\')\n    candidates=[r for r in results if r[\'candidate\'].startswith(\'attnres\')]\n    eligible=[r for r in candidates if r[\'development\'][\'nll\']<=baseline[\'development\'][\'nll\']-.005]\n    winner=max(eligible,key=lambda r:r[\'median_training_tokens_per_second\']) if eligible else min(candidates,key=lambda r:r[\'development\'][\'nll\'])\n    selection={\'winner\':winner[\'candidate\'],\'configuration\':CONFIGS[winner[\'candidate\']],\n        \'quality_selection_condition_passed\':bool(eligible),\'development_seed\':42,\'confirmation_seeds\':[17,123,777],\n        \'target_development_nll\':TARGET,\'rule\':\'Among AttnRes models at least .005 development NLL below baseline6, choose highest throughput; otherwise lowest NLL and flag failure\',\n        \'screen_results_sha256\':digest(HERE/\'screen_results.json\'),\'protocol_sha256\':digest(HERE/\'PROTOCOL.md\')}\n    if (HERE/\'selection.json\').exists():assert json.loads((HERE/\'selection.json\').read_text())==selection\n    dump_json(HERE/\'selection.json\',selection)\n    if DRIVE:backup_tree(HERE,DRIVE/\'phase3\',include_checkpoints=False)\n    print(\'PHASE3_SELECTION_LOCKED\',json.dumps(selection),flush=True)\n\n\ndef confirmation():\n    selection=json.loads((HERE/\'selection.json\').read_text())\n    assert digest(HERE/\'PROTOCOL.md\')==selection[\'protocol_sha256\']\n    assert digest(HERE/\'screen_results.json\')==selection[\'screen_results_sha256\']\n    names=[\'baseline6\',\'baseline5\',selection[\'winner\']]\n    results=[]\n    for index,seed in enumerate(selection[\'confirmation_seeds\']):\n        order=names[index:]+names[:index]\n        for name in order:\n            results.append(train(name,seed,\'confirmation\'))\n            dump_json(HERE/\'confirmation_results.json\',results)\n    if DRIVE:backup_tree(HERE,DRIVE/\'phase3\',include_checkpoints=False)\n    print(\'PHASE3_CONFIRMATION_COMPLETE\',flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'phase\',choices=[\'screen\',\'confirmation\'])\n    args=parser.parse_args()\n    screen() if args.phase==\'screen\' else confirmation()\n', 'ood_evaluation.py': '"""Locked, custom WikiText-2 prefix evaluation after model selection."""\nimport gc\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom datasets import load_dataset\nfrom transformers import AutoTokenizer\nfrom experiment import digest,dump_json,token_blocks\nfrom models import create_model\nfrom runner import evaluate\n\nROOT=Path(__file__).resolve().parent.parent\nHERE=ROOT/\'phase3\'\nOUT=HERE/\'data\'\nDATASET=\'Salesforce/wikitext\'\nREVISION=\'b08601e04326c79dfdd32d625aee71d232d685c3\'\nmanifest_path=OUT/\'wikitext_manifest.json\'\nif manifest_path.exists():\n    manifest=json.loads(manifest_path.read_text())\n    assert digest(OUT/\'wikitext.npy\')==manifest[\'sha256\']\nelse:\n    tokenizer=AutoTokenizer.from_pretrained(str(ROOT/\'vendor/model\'),trust_remote_code=False)\n    ids=[];lines=0\n    stream=load_dataset(DATASET,\'wikitext-2-raw-v1\',revision=REVISION,split=\'test\',streaming=True)\n    for row in stream:\n        text=row[\'text\'].strip()\n        if not text:continue\n        ids.extend(tokenizer.encode(text,add_special_tokens=False)+[tokenizer.eos_token_id]);lines+=1\n        if len(ids)>=128_000:break\n    assert len(ids)>=128_000\n    np.save(OUT/\'wikitext.npy\',np.asarray(ids[:128_000],dtype=np.int32))\n    manifest={\'dataset\':DATASET,\'configuration\':\'wikitext-2-raw-v1\',\'split\':\'test\',\'revision\':REVISION,\n        \'token_cap\':128_000,\'nonempty_lines_consumed\':lines,\'sha256\':digest(OUT/\'wikitext.npy\'),\n        \'tokenizer_sha256\':digest(ROOT/\'vendor/model/tokenizer.json\'),\n        \'protocol_sha256\':digest(HERE/\'OOD_PROTOCOL.md\'),\n        \'packing\':\'Strip nonempty lines; EOS after each; consecutive 129-token blocks; 128 predictions per block\',\n        \'license_tags\':[\'cc-by-sa-3.0\',\'gfdl\'],\'standard_wikitext_protocol\':False}\n    dump_json(manifest_path,manifest)\nprint(\'OOD_DATA_READY\',json.dumps(manifest),flush=True)\nselection=json.loads((HERE/\'selection.json\').read_text())\nconfirmation=json.loads((HERE/\'confirmation_results.json\').read_text())\nassert len(confirmation)==9\nblocks=token_blocks(OUT/\'wikitext.npy\',128)\nresults=[]\nfor row in confirmation:\n    folder=HERE/\'runs/confirmation\'/f"{row[\'candidate\']}_seed{row[\'seed\']}"\n    weights=folder/\'model_weights.pt\'\n    saved=torch.load(weights,map_location=\'cpu\',weights_only=True)\n    model=create_model(row[\'candidate\'],row[\'seed\']).cuda()\n    model.load_state_dict(saved[\'model\'],strict=True)\n    result={\'candidate\':row[\'candidate\'],\'seed\':row[\'seed\'],**evaluate(model,blocks),\n        \'weights_sha256\':digest(weights),\'dataset_manifest_sha256\':digest(manifest_path),\n        \'selection_sha256\':digest(HERE/\'selection.json\'),\'protocol_sha256\':digest(HERE/\'OOD_PROTOCOL.md\')}\n    results.append(result);dump_json(HERE/\'ood_results.json\',results)\n    print(\'PHASE3_OOD_RESULT\',json.dumps(result),flush=True)\n    del model,saved;gc.collect();torch.cuda.empty_cache()\nprint(\'PHASE3_OOD_COMPLETE\',flush=True)\n', 'repeat_timing.py': '"""Short alternating-order repetitions using fixed saved confirmation models."""\nimport gc\nimport json\nfrom pathlib import Path\nimport statistics\nimport time\nimport torch\nfrom experiment import digest,dump_json,environment,token_blocks\nfrom models import create_model\nfrom optimized import TrainingLoss\nfrom benchmark_stage2 import logical_batch\n\nROOT=Path(__file__).resolve().parent.parent\nHERE=ROOT/\'phase3\'\nselection=json.loads((HERE/\'selection.json\').read_text())\nnames=[\'baseline6\',selection[\'winner\']]\ndata=token_blocks(ROOT/\'data/train.npy\',128)\ntorch.set_num_threads(4)\ntorch.backends.cuda.matmul.allow_tf32=False\ntorch.backends.cudnn.allow_tf32=False\nstates={};hashes={}\nfor name in names:\n    path=HERE/\'runs/confirmation\'/f\'{name}_seed777/model_weights.pt\'\n    saved=torch.load(path,map_location=\'cpu\',weights_only=True)\n    model=create_model(name,777).cuda().train();model.load_state_dict(saved[\'model\'])\n    objective=torch.compile(TrainingLoss(model),fullgraph=True,dynamic=False,mode=\'default\')\n    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,betas=(.9,.95),weight_decay=.1,fused=True)\n    states[name]={\'model\':model,\'objective\':objective,\'optimizer\':optimizer,\'scaler\':torch.amp.GradScaler(\'cuda\',init_scale=1024)}\n    hashes[name]=digest(path)\n    del saved\ndef step(name,index):\n    state=states[name];model=state[\'model\'];optimizer=state[\'optimizer\'];scaler=state[\'scaler\']\n    optimizer.zero_grad(set_to_none=True)\n    torch.cuda.synchronize();start=time.perf_counter()\n    x,y=logical_batch(data,888,index)\n    with torch.autocast(\'cuda\',dtype=torch.float16):loss=state[\'objective\'](x,y)\n    scaler.scale(loss).backward();scaler.unscale_(optimizer)\n    torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True)\n    if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite timing loss\')\n    scaler.step(optimizer);scaler.update();torch.cuda.synchronize()\n    return time.perf_counter()-start\nfor name in names:\n    for index in range(8):step(name,index)\nrows=[]\nfor repeat in range(5):\n    order=names if repeat%2==0 else list(reversed(names))\n    speeds={}\n    for name in order:\n        durations=[step(name,8+repeat*50+index) for index in range(50)]\n        speeds[name]=statistics.median(4096/value for value in durations)\n        rows.append({\'repeat\':repeat,\'candidate\':name,\'order\':order.index(name),\n            \'median_training_tokens_per_second\':speeds[name],\'step_seconds\':durations})\n    print(\'PHASE3_TIMING_REPEAT\',repeat,json.dumps(speeds),\n        \'winner_speed_change_percent\',100*(speeds[names[1]]/speeds[\'baseline6\']-1),flush=True)\nbundle={\'rows\':rows,\'source_checkpoint_sha256\':hashes,\'environment\':environment(\'cuda\'),\n    \'protocol_sha256\':digest(HERE/\'TIMING_PROTOCOL.md\'),\'source_sha256\':digest(__file__),\n    \'winner\':names[1],\'timing_only\':True,\'saved_weights_unchanged\':True}\nfor name in names:\n    assert digest(HERE/\'runs/confirmation\'/f\'{name}_seed777/model_weights.pt\')==hashes[name]\ndump_json(HERE/\'repeated_timing.json\',bundle)\nprint(\'PHASE3_REPEATED_TIMING_COMPLETE\',flush=True)\n', 'analyze.py': '"""Analyze the fixed development screen, confirmation, and independent-domain check."""\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport statistics as stats\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nHERE=Path(__file__).resolve().parent\nOUT=HERE/\'analysis\'\nOUT.mkdir(exist_ok=True)\nselection=json.loads((HERE/\'selection.json\').read_text())\nscreen=json.loads((HERE/\'screen_results.json\').read_text())\nconfirmation=json.loads((HERE/\'confirmation_results.json\').read_text())\nood=json.loads((HERE/\'ood_results.json\').read_text())\nholdout=json.loads((HERE/\'data/holdout_manifest.json\').read_text())\nood_manifest=json.loads((HERE/\'data/wikitext_manifest.json\').read_text())\nrepeated=json.loads((HERE/\'repeated_timing.json\').read_text())\nassert len(screen)==6 and len(confirmation)==9 and len(ood)==9\nwinner=selection[\'winner\'];names=[\'baseline6\',\'baseline5\',winner]\nseeds=selection[\'confirmation_seeds\']\nassert {(r[\'candidate\'],r[\'seed\']) for r in confirmation}=={(name,seed) for name in names for seed in seeds}\nassert all(r[\'steps\']==1000 and r[\'training_tokens\']==4_096_000 for r in screen+confirmation)\nconfigs=[];histories=[];backups=[]\nfor row in screen+confirmation:\n    folder=HERE/\'runs\'/row[\'phase\']/f"{row[\'candidate\']}_seed{row[\'seed\']}"\n    config=json.loads((folder/\'config.json\').read_text())\n    configs.append(config)\n    for source,expected in config[\'source_hashes\'].items():\n        actual=hashlib.sha256((HERE/source).read_bytes()).hexdigest()\n        if actual!=expected:\n            # The signed run hashes remain unchanged. Only a recorded editorial\n            # revision of the non-executable protocol is accepted here.\n            revisions=json.loads((HERE/\'document_revisions.json\').read_text())\n            revision=next((r for r in revisions[\'files\'] if r[\'path\']==\'phase3/\'+source),None)\n            assert source==\'PROTOCOL.md\' and revision is not None,source\n            assert revision[\'original_sha256\']==expected and revision[\'revised_sha256\']==actual,source\n    if row[\'phase\']==\'confirmation\':\n        assert config[\'selection_sha256\']==hashlib.sha256((HERE/\'selection.json\').read_bytes()).hexdigest()\n    for point in json.loads((folder/\'history.json\').read_text()):\n        histories.append({\'candidate\':row[\'candidate\'],\'seed\':row[\'seed\'],\'phase\':row[\'phase\'],**point})\n    backups.extend(json.loads((folder/\'backup_verification.json\').read_text()))\nassert len({json.dumps(c[\'source_hashes\'],sort_keys=True) for c in configs})==1\nassert len({json.dumps(c[\'environment\'],sort_keys=True) for c in configs})==1\nassert all(r[\'resume_step_this_session\']==0 for r in confirmation),\'Resumed timings require an explicit separate analysis\'\nby_key={(r[\'candidate\'],r[\'seed\']):r for r in confirmation}\nood_by_key={(r[\'candidate\'],r[\'seed\']):r for r in ood}\nsummary={}\nfor name in names:\n    rows=[r for r in confirmation if r[\'candidate\']==name]\n    values={\n        \'heldout_nll\':[r[\'heldout\'][\'nll\'] for r in rows],\n        \'heldout_perplexity\':[r[\'heldout\'][\'perplexity\'] for r in rows],\n        \'development_nll\':[r[\'development\'][\'nll\'] for r in rows],\n        \'training_tokens_per_second\':[r[\'median_training_tokens_per_second\'] for r in rows],\n        \'wikitext_nll\':[ood_by_key[name,r[\'seed\']][\'nll\'] for r in rows],\n        \'wikitext_perplexity\':[ood_by_key[name,r[\'seed\']][\'perplexity\'] for r in rows],\n        \'target_wall_seconds\':[r[\'first_observed_target_crossing\'][\'wall_seconds\'] for r in rows if r[\'first_observed_target_crossing\']],\n        \'target_training_seconds\':[r[\'first_observed_target_crossing\'][\'training_seconds\'] for r in rows if r[\'first_observed_target_crossing\']],\n        \'full_run_wall_seconds\':[r[\'active_run_wall_seconds_including_setup_compile_eval_checkpoint_io\'] for r in rows],\n        \'peak_gpu_mib\':[r[\'peak_allocated_gpu_bytes_this_session\']/2**20 for r in rows]}\n    summary[name]={\'parameters\':rows[0][\'parameters\'],\n        **{key:{\'count\':len(v),\'mean\':stats.mean(v) if v else None,\'sample_sd\':stats.stdev(v) if len(v)>1 else None} for key,v in values.items()}}\npaired=[]\nfor seed in seeds:\n    a,b,c=by_key[\'baseline6\',seed],by_key[winner,seed],by_key[\'baseline5\',seed]\n    target_a,target_b=a[\'first_observed_target_crossing\'],b[\'first_observed_target_crossing\']\n    paired.append({\'seed\':seed,\n        \'heldout_nll_difference\':b[\'heldout\'][\'nll\']-a[\'heldout\'][\'nll\'],\n        \'heldout_perplexity_reduction_percent\':100*(1-b[\'heldout\'][\'perplexity\']/a[\'heldout\'][\'perplexity\']),\n        \'throughput_change_percent\':100*(b[\'median_training_tokens_per_second\']/a[\'median_training_tokens_per_second\']-1),\n        \'target_wall_reduction_percent\':100*(1-target_b[\'wall_seconds\']/target_a[\'wall_seconds\']) if target_a and target_b else None,\n        \'target_training_time_reduction_percent\':100*(1-target_b[\'training_seconds\']/target_a[\'training_seconds\']) if target_a and target_b else None,\n        \'wikitext_nll_difference\':ood_by_key[winner,seed][\'nll\']-ood_by_key[\'baseline6\',seed][\'nll\'],\n        \'wikitext_perplexity_reduction_percent\':100*(1-ood_by_key[winner,seed][\'perplexity\']/ood_by_key[\'baseline6\',seed][\'perplexity\']),\n        \'same_depth5_nll_difference\':b[\'heldout\'][\'nll\']-c[\'heldout\'][\'nll\'] if selection[\'configuration\'][\'layers\']==5 else None,\n        \'same_depth5_throughput_change_percent\':100*(b[\'median_training_tokens_per_second\']/c[\'median_training_tokens_per_second\']-1) if selection[\'configuration\'][\'layers\']==5 else None})\nquality_all=all(r[\'heldout_nll_difference\']<0 for r in paired)\nspeed_all=all(r[\'throughput_change_percent\']>0 for r in paired)\nood_all=all(r[\'wikitext_nll_difference\']<0 for r in paired)\ncriteria={\'lower_tinystories_nll_every_seed\':quality_all,\'higher_raw_throughput_every_seed\':speed_all,\n    \'primary_local_faster_and_stronger_criterion_met\':quality_all and speed_all,\n    \'lower_custom_wikitext_nll_every_seed\':ood_all,\n    \'lower_target_wall_every_seed\':all(r[\'target_wall_reduction_percent\'] is not None and r[\'target_wall_reduction_percent\']>0 for r in paired)}\nbundle={\'selection\':selection,\'criteria\':criteria,\'summary\':summary,\'paired\':paired,\'screen\':screen,\n    \'confirmation\':confirmation,\'ood\':ood,\'configs\':configs,\'holdout_manifest\':holdout,\'ood_manifest\':ood_manifest,\n    \'repeated_timing\':repeated}\n(OUT/\'phase3_results.json\').write_text(json.dumps(bundle,indent=2),encoding=\'utf-8\')\nflat=[]\nfor r in confirmation:\n    flat.append({\'candidate\':r[\'candidate\'],\'seed\':r[\'seed\'],\'parameters\':r[\'parameters\'],\n        \'heldout_nll\':r[\'heldout\'][\'nll\'],\'heldout_perplexity\':r[\'heldout\'][\'perplexity\'],\n        \'wikitext_nll\':ood_by_key[r[\'candidate\'],r[\'seed\']][\'nll\'],\n        \'wikitext_perplexity\':ood_by_key[r[\'candidate\'],r[\'seed\']][\'perplexity\'],\n        \'training_tokens_per_second\':r[\'median_training_tokens_per_second\'],\n        \'target_step\':r[\'first_observed_target_crossing\'][\'step\'] if r[\'first_observed_target_crossing\'] else None,\n        \'target_wall_seconds\':r[\'first_observed_target_crossing\'][\'wall_seconds\'] if r[\'first_observed_target_crossing\'] else None,\n        \'full_run_wall_seconds\':r[\'active_run_wall_seconds_including_setup_compile_eval_checkpoint_io\']})\nfor name,rows in [(\'confirmation_results.csv\',flat),(\'paired_comparisons.csv\',paired),(\'learning_curves.csv\',histories)]:\n    with (OUT/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as handle:\n        writer=csv.DictWriter(handle,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)\ngain=stats.mean(r[\'heldout_perplexity_reduction_percent\'] for r in paired)\nspeed=stats.mean(r[\'throughput_change_percent\'] for r in paired)\nood_gain=stats.mean(r[\'wikitext_perplexity_reduction_percent\'] for r in paired)\nwall_values=[r[\'target_wall_reduction_percent\'] for r in paired if r[\'target_wall_reduction_percent\'] is not None]\nwall_gain=stats.mean(wall_values) if wall_values else None\nreduction=100*(1-summary[winner][\'parameters\']/summary[\'baseline6\'][\'parameters\'])\nstatus=\'MET within the stated small-model TinyStories comparison\' if criteria[\'primary_local_faster_and_stronger_criterion_met\'] else \'NOT MET on the locked three-seed comparison\'\nlines=[\'# mini-k3: lower-cost attention residuals and compact-model confirmation\',\'\',\n    \'Date: 18 October 2026. Author: Fazal Hameed.\',\'\',\n    \'## Outcome\',\'\',\n    f"Selected model: **{winner}**, {summary[winner][\'parameters\']:,} parameters ({reduction:.2f}% fewer than baseline6). "\n    f\'The mean paired heldout-perplexity reduction was **{gain:.2f}%** and the mean paired raw-throughput change was **{speed:+.2f}%**. \'\n    f\'The internally fixed faster-and-stronger criterion was **{status}**.\',\'\',\n    f\'The additional custom WikiText-2 test-prefix perplexity change was a {ood_gain:+.2f}% reduction \'\n    f\'(positive means better). Lower WikiText NLL in every seed: {ood_all}. \'\n    \'These results do not establish general LLM capability, Chinese ability or assistant quality.\',\'\',\n    \'## Experiment design\',\'\',\n    \'The six-layer pilot MiniMind baseline and five-layer ordinary-residual control were compared with four AttnRes \'\n    \'candidates: five/six layers and blocks of four/twelve sublayers. All retain width 384, six query/two KV heads, \'\n    \'vocabulary 6,400, the original tokenizer, causal grouped-query attention, rotary positions, dense feed-forward \'\n    \'layers and FP32 residual/block accumulation under FP16 linear autocast. The models train from scratch. \'\n    \'MiniMind is pinned at `f659b55761b754d306bd140573493a6543cafd7f` (Apache-2.0).\',\'\',\n    \'Block Attention Residuals follows the depth-mixing idea in [Kimi K3, section 2.2]\'\n    \'(https://arxiv.org/html/2607.24653v2#S2.SS2). The new implementation stores completed block states in tuples \'\n    \'rather than repeatedly stacking full hidden-state tensors. It caches each completed state’s inverse RMS \'\n    \'within the differentiable forward computation. With two source scores s0/s1 and values v0/v1, \'\n    \'the second softmax weight equals sigmoid(s1-s0); the mixture is v0 + gate*(v1-v0). \'\n    \'More than two sources use small score stacks and \'\n    \'weighted sums. The sigmoid identity is algebraically equivalent, with floating-point roundoff differences.\',\'\',\n    \'Larger blocks reduce the number of completed depth sources, while the mixer is still evaluated before each \'\n    \'attention/MLP and at the output. A block size of twelve exceeds a five-layer model’s ten sublayers; \'\n    \'its final mixer includes the unfinished partial sum. Compactness is a real architecture change, and \'\n    \'the comparison is not parameter-matched. The five-layer ordinary control separates depth reduction \'\n    \'from adding AttnRes. Matching-depth models share identical initial weights; different-depth models \'\n    \'use the same seed but do not assert all initial tensors are identical. No KDA, LatentMoE, SiTU-GLU or Muon \'\n    \'is implemented. No KV-cache or decoding-speed claim is made.\',\'\',\n    \'## Development screen and selection\',\'\',\n    \'An internal protocol was written before the new training. Seed 42 was used only for development. \'\n    \'Each of six candidates trained for 1,000 updates. Among AttnRes candidates whose endpoint development \'\n    \'NLL beat baseline6 by at least 0.005, the highest measured steady training throughput was selected. \'\n    \'If no candidate passed, the lowest-NLL AttnRes candidate would be selected and the condition flagged as \'\n    \'failed. Selection never used new confirmation holdout or WikiText scores. This was an internal \'\n    \'prospective plan, not an externally registered study.\',\'\',\n    \'| Candidate | Parameters | Development NLL | Training tokens/s | First step, s |\',\n    \'| --- | ---: | ---: | ---: | ---: |\']\nfor r in screen:\n    lines.append(f"| {r[\'candidate\']} | {r[\'parameters\']:,} | {r[\'development\'][\'nll\']:.4f} | "\n        f"{r[\'median_training_tokens_per_second\']:.0f} | {r[\'first_step_seconds_including_compilation\']:.2f} |")\nlines+=[\'\',f"Selected: `{winner}`. Quality selection condition passed: {selection[\'quality_selection_condition_passed\']}.",\'\',\n    \'## Locked training and heldout evaluation\',\'\',\n    \'- Three confirmation seeds: 17, 123 and 777. Seeds 17/123 occurred in earlier work, but were not used to select \'\n    \'this phase’s winner; 777 is new. The development screen uses seed 42 only.\',\n    \'- Baseline6, baseline5 and the locked winner run at every seed, with execution order rotated.\',\n    \'- Free Tesla T4 (14.56 GiB), Python 3.13.15, PyTorch 2.11.0+cu130, Transformers 4.57.6.\',\n    \'- Physical/logical batch 32, 128 predicted tokens per sequence: 4,096 predicted tokens per update.\',\n    \'- 1,000 updates and 4,096,000 processed training tokens per run. Same seed/step-dependent batch stream.\',\n    \'- Fused AdamW, peak LR 3e-4, 50-update warmup and cosine decay toward 10% peak, betas (.9,.95), weight decay .1.\',\n    \'- FP32 parameters and residual/block states; FP16 linear autocast; initial loss scale 1,024; clip norm 1.0.\',\n    \'- Full-graph `torch.compile`, default mode, fixed shapes. All candidates pass compiled/eager scaled-gradient checks.\',\n    \'- Development NLL evaluated every 50 updates on the same 32,768 predicted tokens. No hyperparameter sweep.\',\n    \'- Nonfinite loss/gradients halt training; no silently skipped updates or unequal token budgets.\',\'\',\n    \'Training uses the same pinned, exact-text-deduplicated two-million-token TinyStories cache. It is sampled \'\n    \'with replacement, so processed-token count is not a distinct-token count. Dataset revision \'\n    \'`f54c09fd23315a6f9c86f9dc80f725de7d8f9c64`, CDLA-Sharing-1.0. The training cache, original validation \'\n    \'prefix and phase-two heldout prefix were reconstructed and checked for exact token equality. \'\n    f"A new partition uses {holdout[\'new_holdout_stories\']} subsequent complete validation stories, "\n    f"128,000 cached tokens and {confirmation[0][\'heldout\'][\'tokens\']:,} predicted tokens in full packed blocks. "\n    \'Complete selected train, original validation and earlier heldout stories are excluded by exact text hash. \'\n    \'Near-duplicate/paraphrase removal was not performed. The new partition is not an official test split \'\n    \'or a new domain; it is evaluated only at the fixed confirmation endpoint.\',\'\',\n    \'Perplexity is exp(mean next-token NLL in nats/token). Packed blocks can cross story boundaries and \'\n    \'do not mask attention at EOS. Both models use the same packing, tokenizer and causal mask.\',\'\',\n    \'## Three-seed confirmation results\',\'\',\n    \'| Seed | Candidate | Heldout NLL | Heldout perplexity | Training tokens/s | Target step | Target wall, s |\',\n    \'| --- | --- | ---: | ---: | ---: | ---: | ---: |\']\nfor row in flat:\n    step=\'not reached\' if row[\'target_step\'] is None else str(row[\'target_step\'])\n    wall=\'not reached\' if row[\'target_wall_seconds\'] is None else f"{row[\'target_wall_seconds\']:.2f}"\n    lines.append(f"| {row[\'seed\']} | {row[\'candidate\']} | {row[\'heldout_nll\']:.4f} | {row[\'heldout_perplexity\']:.3f} | "\n        f"{row[\'training_tokens_per_second\']:.0f} | {step} | {wall} |")\nlines+=[\'\',\'Means ± sample SD across three confirmation seeds:\',\'\',\n    \'| Candidate | Heldout perplexity | Training tokens/s | Target wall, s | Peak GPU MiB |\',\n    \'| --- | ---: | ---: | ---: | ---: |\']\nfor name in names:\n    s=summary[name]\n    def fmt(key,places):\n        v=s[key]\n        if v[\'count\']!=3:return f"{v[\'count\']}/3 reached"\n        return f"{v[\'mean\']:.{places}f} ± {v[\'sample_sd\']:.{places}f}"\n    lines.append(f"| {name} | {fmt(\'heldout_perplexity\',3)} | {fmt(\'training_tokens_per_second\',0)} | {fmt(\'target_wall_seconds\',2)} | {fmt(\'peak_gpu_mib\',1)} |")\nlines+=[\'\',\'Paired winner changes against baseline6 (positive percentages mean improvement):\',\'\',\n    \'| Seed | Heldout perplexity reduction | Throughput increase | Target wall reduction | WikiText perplexity reduction |\',\n    \'| --- | ---: | ---: | ---: | ---: |\']\nfor p in paired:\n    target=\'not comparable\' if p[\'target_wall_reduction_percent\'] is None else f"{p[\'target_wall_reduction_percent\']:+.2f}%"\n    lines.append(f"| {p[\'seed\']} | {p[\'heldout_perplexity_reduction_percent\']:+.2f}% | {p[\'throughput_change_percent\']:+.2f}% | "\n        f"{target} | {p[\'wikitext_perplexity_reduction_percent\']:+.2f}% |")\nlines+=[\'\',\'## Repeated timing validation\',\'\',\n    \'A separate timing-only check loads the fixed seed-777 confirmation checkpoints, uses fresh AdamW/scaler \'\n    \'states, warms up eight updates per model, then measures five repetitions of 50 updates each. \'\n    \'The execution order alternates. Data seed 888, LR 1e-4 and the same precision/compiler settings \'\n    \'are fixed. The temporary benchmark updates do not modify saved weights and do not produce quality scores. \'\n    \'This check reduces the separation between speed comparisons; it does not replace the primary criterion.\',\'\',\n    \'| Repetition | Baseline6 tokens/s | Winner tokens/s | Winner speed change |\',\n    \'| --- | ---: | ---: | ---: |\']\nrepeat_changes=[]\nfor repeat in range(5):\n    a=next(r for r in repeated[\'rows\'] if r[\'repeat\']==repeat and r[\'candidate\']==\'baseline6\')[\'median_training_tokens_per_second\']\n    b=next(r for r in repeated[\'rows\'] if r[\'repeat\']==repeat and r[\'candidate\']==winner)[\'median_training_tokens_per_second\']\n    change=100*(b/a-1);repeat_changes.append(change)\n    lines.append(f\'| {repeat+1} | {a:.0f} | {b:.0f} | {change:+.2f}% |\')\nlines+=[\'\',f\'Mean paired repeated-benchmark speed change: {stats.mean(repeat_changes):+.2f}%. \'\n    f\'Higher winner throughput in all five repetitions: {all(v>0 for v in repeat_changes)}.\',\'\',\n    \'## Time to a fixed prediction quality\',\'\',\n    \'The fixed target is development NLL <= 2.10, chosen from the earlier phase before these runs. \'\n    \'The first observed crossing is measured at 50-update intervals; it is not interpolated or an exact \'\n    \'earliest crossing. Cumulative training-step time and wall time are both retained. Wall time includes \'\n    \'model creation, compiler-cache loading/compilation, development evaluation and any runtime checkpoints \'\n    \'written before the crossing. It excludes dependency/data preparation and final model export/Drive backup. \'\n    \'No Drive checkpoint writes occur inside the timed training run. These are training-time measurements \'\n    \'and cannot be described as inference latency.\',\'\',\n    (f\'Mean paired target wall-time reduction: {wall_gain:+.2f}%.\' if wall_gain is not None else \'No paired target wall-time comparison was possible.\'),\n    \'Faster time to this one quality threshold is a separate metric. It cannot replace a failed raw \'\n    \'throughput result or establish performance at every quality target.\',\'\',\n    \'Steady throughput includes synchronized batch construction/transfer, forward/backward, scaling, clipping \'\n    \'and optimizer work after the first eight updates; it excludes evaluation/checkpoint IO. Compiler caches \'\n    \'were primed by correctness checks and development runs. First-step times are recorded but are not a \'\n    \'controlled empty-cache compilation experiment. Shared free hardware, thermal/load variation, a single \'\n    \'session and limited order rotation restrict small timing differences. Peak memory is PyTorch allocated \'\n    \'memory, not reserved or total-device use.\',\'\',\n    \'## Additional-domain check\',\'\',\n    \'After selection and confirmation training, all nine checkpoints are evaluated on the same custom \'\n    \'[WikiText-2 raw test prefix](https://huggingface.co/datasets/Salesforce/wikitext), pinned at \'\n    \'`b08601e04326c79dfdd32d625aee71d232d685c3`. The unchanged MiniMind tokenizer encodes stripped nonempty \'\n    \'lines with EOS after each; the first 128,000 tokens are packed as above. \'\n    f"This evaluates {ood[0][\'tokens\']:,} predicted tokens. License tags on the source card are CC-BY-SA-3.0 and GFDL. "\n    \'The additional-domain plan was fixed before selection and inspecting development endpoint results. \'\n    \'This is not the standard published WikiText perplexity protocol and is not comparable with numbers \'\n    \'using another tokenizer, context or line handling. No WikiText training or tuning was performed.\',\'\',\n    \'| Seed | Candidate | Custom WikiText NLL | Custom WikiText perplexity |\',\n    \'| --- | --- | ---: | ---: |\']\nfor r in ood:lines.append(f"| {r[\'seed\']} | {r[\'candidate\']} | {r[\'nll\']:.4f} | {r[\'perplexity\']:.3f} |")\nlines+=[\'\',\'## Interpretation and limits\',\'\',\n    f\'Primary condition: lower TinyStories heldout NLL in every seed = {quality_all}; \'\n    f\'higher throughput in every seed = {speed_all}. \'\n    f\'Additional-domain lower NLL in every seed = {ood_all}. \'\n    \'A local success supports only this compact-model, fixed-budget comparison on the measured hardware. \'\n    \'It does not show that adding AttnRes to an otherwise identical model has zero cost. \'\n    \'The same-depth ordinary control and development candidates reveal that distinction. \'\n    \'A larger model study, more domains, downstream tasks, cached inference and repeated timing sessions \'\n    \'remain. Three seeds are not proof of general superiority.\',\'\',\n    \'The custom WikiText perplexities are very high in absolute terms: all these story-trained models \'\n    \'remain poor encyclopedic language models. A mixed additional-domain result prevents a claim of \'\n    \'consistent superiority across domains. Phase-three and phase-two heldout sets differ, so their \'\n    \'percentage improvements should not be compared as if measured on identical text.\',\'\',\n    \'Phase one’s original paths accumulated residuals at different precision. Phase two corrected that \'\n    \'confound and found a quality gain with a speed cost. Phase three preserves the precision control, \'\n    \'changes the implementation/block size, and explicitly tests smaller capacity. Earlier sources and \'\n    \'measurements remain intact; the new result must not be substituted retrospectively into old runs.\',\'\',\n    \'## Correctness and artifacts\',\'\',\n    \'CPU/reference checks cover tuple/reference logits and gradients with nonzero queries, causality, \'\n    \'matching-depth initial weights, and FP32 states under GPU autocast. All six candidates passed \'\n    \'compiled/eager losses and scaled-gradient checks at batch 32/context 128. These correctness tests \'\n    \'are distinct from model-quality results. Configuration, data/code hashes, histories, per-step timing \'\n    \'and individual confirmation scores are retained. No GPU interrupted/resumed trajectory-equivalence \'\n    \'test was performed for this runner.\',\'\',\n    \'Full resumable checkpoints were written every 250 updates in the original Colab runtime. \'\n    \'That runtime expired after the experiments finished; its temporary optimizer checkpoints were lost. New completed \'\n    \'model-only files, configs, results and logs are backed up with SHA-256 verification under \'\n    \'`My Drive/mini-k3/2026-10-18/phase3`. These persistent model exports omit AdamW/scaler state and \'\n    \'cannot resume the exact optimizer trajectory after runtime loss. They can reproduce evaluation \'\n    \'with the pinned source. Local bundles contain code, measured evidence and reports; model and \'\n    \'cached-token bytes remain in Drive/the runtime.\',\'\',\n    \'\']\n(OUT/\'PHASE3_REPORT.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\nfig=plt.figure(figsize=(10,6.2))\ngrid=fig.add_gridspec(2,2)\naxes=[fig.add_subplot(grid[0,:]),fig.add_subplot(grid[1,0]),fig.add_subplot(grid[1,1])]\npalette={\'baseline6\':\'#3659a5\',\'baseline5\':\'#6d8583\',winner:\'#d46423\'}\nfor name in names:\n    points=sorted({r[\'step\'] for r in histories if r[\'phase\']==\'confirmation\' and r[\'candidate\']==name})\n    values=[[r[\'nll\'] for r in histories if r[\'phase\']==\'confirmation\' and r[\'candidate\']==name and r[\'step\']==s] for s in points]\n    means=[stats.mean(v) for v in values]\n    axes[0].plot([s*4096/1e6 for s in points],means,label=name,color=palette[name])\n    axes[1].scatter([summary[name][\'training_tokens_per_second\'][\'mean\']/1000],\n        [summary[name][\'heldout_perplexity\'][\'mean\']],label=name,color=palette[name],s=70)\n    targets=[by_key[name,seed][\'first_observed_target_crossing\'] for seed in seeds]\n    axes[2].scatter([name for t in targets if t],[t[\'wall_seconds\'] for t in targets if t],color=palette[name],s=35)\naxes[0].axhline(2.1,color=\'#777777\',ls=\'--\',lw=.8)\naxes[0].set(xlabel=\'Training tokens (millions)\',ylabel=\'Development NLL\',title=\'Three-seed mean learning curves\')\naxes[0].legend(fontsize=7);axes[0].grid(alpha=.2)\naxes[1].set(xlabel=\'Training throughput (thousand tokens/s)\',ylabel=\'Heldout perplexity (lower is better)\',title=\'Quality and raw throughput\')\naxes[1].legend(fontsize=7);axes[1].grid(alpha=.2)\naxes[2].set(ylabel=\'First observed target wall time (s)\',title=\'Fixed development NLL target <= 2.10\')\naxes[2].tick_params(axis=\'x\',labelrotation=20);axes[2].grid(axis=\'y\',alpha=.2)\nfig.tight_layout();fig.savefig(OUT/\'phase3_results.png\',dpi=180);plt.close(fig)\nprint(\'PHASE3_ANALYSIS\',json.dumps({\'winner\':winner,\'criteria\':criteria,\'summary\':summary,\'paired\':paired}),flush=True)\nprint(\'PHASE3_ANALYSIS_COMPLETE\',OUT/\'PHASE3_REPORT.md\',flush=True)\n', 'export_logs.py': '"""Export measured evidence and exact source through visible notebook output."""\nimport base64\nimport hashlib\nimport io\nfrom pathlib import Path\nimport zipfile\n\nROOT=Path(__file__).resolve().parent.parent\nbuffer=io.BytesIO()\nwith zipfile.ZipFile(buffer,\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n    for path in sorted((ROOT/\'phase3\').rglob(\'*\')):\n        if not path.is_file() or \'__pycache__\' in path.parts:continue\n        if path.suffix in (\'.json\',\'.csv\',\'.md\',\'.py\'):\n            archive.write(path,path.relative_to(ROOT))\nblob=buffer.getvalue();encoded=base64.b64encode(blob).decode(\'ascii\')\nchunks=[encoded[i:i+6000] for i in range(0,len(encoded),6000)]\nprint(\'P3LOG_SIZE:\'+str(len(blob)))\nprint(\'P3LOG_SHA256:\'+hashlib.sha256(blob).hexdigest())\nprint(\'P3LOG_CHUNK_COUNT:\'+str(len(chunks)))\nfor index,chunk in enumerate(chunks):print(f\'P3LOG_CHUNK_{index}:\'+chunk)\n', 'PROTOCOL.md': "# Phase-three protocol, fixed before new training\n\nDate: 18 October 2026. Aim: reduce the Block AttnRes cost and investigate a\nsmaller model, with a fair optimized baseline and explicitly defined quality.\n\n## Candidates and selection\n\nSix candidates: ordinary six-layer and five-layer MiniMind; tuple-based AttnRes\nwith six/five layers and block sizes four/twelve sublayers. All use hidden width\n384, MiniMind's original tokenizer, sequence 128, FP32 residual and block states,\nFP16 linear autocast, fused AdamW and the same compilation settings. Block twelve\ncan exceed the ten sublayers of the five-layer model; its final mixer includes\nthe unfinished partial block. This is a documented configuration, not a full K3\nreproduction. Smaller models have fewer parameters and are explicitly labeled.\n\nThe tuple implementation avoids stacking full hidden-state sources and uses the\ntwo-source sigmoid identity. Reference outputs/gradients, causal behavior,\nshared initialization and mixed-precision states must pass correctness checks.\nCompiled/eager gradients use the trainer's loss scaling and unchanged tolerance.\n\nSeed 42 is for development only: all six candidates train from scratch for\n1,000 updates (4,096,000 predicted tokens each). This uses the existing train and\ndevelopment-validation caches. No new heldout scores are computed during this\nscreen. Among AttnRes candidates whose endpoint development NLL is at least\n0.005 below baseline6, select the highest measured steady throughput. If none\nqualifies, select the lowest development NLL and record that the quality\nselection condition failed. Do not select using the confirmation holdout.\n\n## Locked confirmation\n\nAfter selection, lock its configuration. Run baseline6, baseline5 and the winner\nat fresh confirmation seeds 17, 123 and 777. Seeds 17/123 appeared in phase two\nbut are unused in this phase's selection; 777 is new to the study. The primary\nthree-seed confirmation is independent of the seed-42 development screen.\nUse rotated execution order across the three seeds. All runs receive matching\ninitial shared weights at a given depth and matching sampled training blocks.\n\nThe fixed budget is 1,000 updates, physical/logical batch 32, sequence 128,\nAdamW fused, LR 3e-4, 50-update warmup/cosine decay, betas (.9,.95), weight decay\n.1, gradient clipping 1.0, initial loss scale 1,024. No skipped nonfinite steps.\nEvaluate development NLL every 50 updates. These settings stay unchanged for\nall configurations; a new experiment is needed for a new setting.\n\nPrepare a new 128,000-token partition from subsequent TinyStories validation\nstories, excluding complete stories used in training, earlier validation and\nphase-two holdout by exact text hash. Reconstruct and verify all preceding\ntoken caches. The new partition remains part of the official validation split,\nnot an official test split or independent domain. Evaluate it only at each\nconfirmation run's fixed endpoint; never use it to choose the winner.\n\n## Speed and success criteria\n\n- Raw training throughput: median synchronized predicted tokens/s after eight\n  warmup updates, excluding evaluation/checkpoint IO. Use the same engine for all.\n- Fixed quality target: development NLL <= 2.10. Record the first observed\n  crossing at 50-update evaluation intervals, without interpolation. Report both\n  cumulative training-step seconds and wall seconds including model setup,\n  compiler-cache loading/compilation, evaluation and runtime checkpoint writes.\n  Data/dependency preparation and final Drive backup are outside this timing.\n- Steady speed is not cold-start or cached decoding speed. Record first-step\n  time, parameter count, peak allocated memory and compiler-cache limitations.\n- A local faster-and-stronger result requires the selected model to have lower\n  confirmation-holdout NLL and higher raw throughput than baseline6 in every\n  confirmation seed. A faster time to target is a separate claim and cannot\n  substitute for a failed raw-throughput comparison. Report means, sample SD,\n  individual pairs and any failures, without a general-capability claim.\n\n## Preservation\n\nKeep earlier sources and scores intact. Save full resumable checkpoints in the\nruntime every 250 updates. Back up new completed model weights, configuration,\nlogs and scores to the approved Drive folder, using SHA-256 verification. To\nconserve Drive space, these new persistent model files omit optimizer/scaler\nstate; they support inference/evaluation but cannot resume the exact optimizer\ntrajectory after runtime loss. Retain that limitation in the report.\n\n", 'OOD_PROTOCOL.md': '# Additional-domain evaluation, fixed before development results\n\n18 October 2026. This addendum is fixed while the first development model is\nstill running, before any candidate is selected or any additional-domain score\nis computed. It does not change the candidate-selection rule or primary criteria\nin PROTOCOL.md.\n\nAfter the selection is locked and confirmation training has finished, evaluate\nthe nine confirmation checkpoints on a fixed 128,000-token prefix of\n`Salesforce/wikitext`, configuration `wikitext-2-raw-v1`, official `test` split,\nrevision `b08601e04326c79dfdd32d625aee71d232d685c3`.\n\nUse the unchanged MiniMind tokenizer. Strip each source line, skip empty lines,\nappend EOS after each nonempty line, and concatenate tokens in dataset order.\nUse non-overlapping packed 129-token blocks, predicting 128 tokens per block.\nDo not fine-tune, select candidates, alter budgets or retry new architectures\nusing this score. Report the same NLL and perplexity metrics for every model.\n\nThis is a custom tokenized test-prefix evaluation, not the published standard\nWikiText-2 perplexity protocol. It provides a limited domain-shift check on\nencyclopedic English after training on simple stories. Tokenizer, line-level EOS\nand context packing restrict external comparisons. Prefix selection need not\nrepresent the full dataset. No general reasoning or Chinese-language claim is\nsupported. Report any regression even if the primary TinyStories result wins.\n\nSources: https://huggingface.co/datasets/Salesforce/wikitext and the pinned\ncommit at https://huggingface.co/datasets/Salesforce/wikitext/commit/b08601e04326c79dfdd32d625aee71d232d685c3.\nDataset card license tags: CC-BY-SA-3.0 and GFDL. Keep attribution with any\nredistributed token cache. The local result bundle excludes cached token bytes.\n', 'TIMING_PROTOCOL.md': '# Repeated timing validation\n\nFixed before inspecting confirmation results. After confirmation and the\nadditional-domain evaluation, load baseline6 and the locked winner from seed\n777, the preselected last confirmation seed. Use fresh optimizer/scaler states\nand fixed data seed 888. This timing-only test does not produce quality scores\nand does not modify the saved model files.\n\nUse the same default full-graph compiled FP16/FP32 training path, batch 32 and\nsequence 128. Warm up each model for eight updates, then measure five repetitions\nof 50 updates per model. Reverse model execution order in alternate repetitions.\nUse matching training blocks within each repetition. AdamW fused LR 1e-4,\nbetas (.9,.95), weight decay .1, gradient norm clip 1, loss scale initially 1,024.\nInclude batch construction, transfer and complete synchronized optimizer work.\n\nReport per-repetition median throughput, paired speed change, overall step\ntimings and source checkpoint hashes. Do not replace an unfavorable primary\nconfirmation criterion with this additional result. This check reduces time\nseparation between comparisons, but shared hardware remains a limitation.\n', 'document_revisions.json': '{\r\n  "date": "2026-10-18",\r\n  "scope": "Editorial wording revision; all original run configs, source hashes, histories and measured result files are retained unchanged. The protocol revision removes only the closing nontechnical paragraph.",\r\n  "files": [\r\n    {\r\n      "path": "phase3/analyze.py",\r\n      "original_sha256": "0435e2cefe5ddeba8b4483c10cc4069f0255b5b5f3e0fcfdd05d149cd8c11797",\r\n      "revised_sha256": "352c0f2c9f32f3e081c999d5f68123f871c1de8c0620ba8e82a91c3ccadb0c9c",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    },\r\n    {\r\n      "path": "phase3/PROTOCOL.md",\r\n      "original_sha256": "d44508abb46d5876a0e12921f7fc6f2ad34c3a5233a231c1274d5fd4dc1bbbfd",\r\n      "revised_sha256": "e5fbea8cc56e1d850d323cd01df59d6bab6cfef928371302e68013f93cb5fad2",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    },\r\n    {\r\n      "path": "phase3/finalize_local.py",\r\n      "original_sha256": "f11e51570231b22b62c26fa80243e7898e8b8f299cc9972456883659257e9876",\r\n      "revised_sha256": "1a9fa8c98a597e48be4db725de49ddc010d02ab6366eecd202d3b870206bf142",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    },\r\n    {\r\n      "path": "phase2/build_pdf.py",\r\n      "original_sha256": "5276f0da3e75d85c7ea819f69c3501cc989424ab6e90bc4c5a4f831dcdc05ffd",\r\n      "revised_sha256": "b4484a242bf9c65f368124cebdef001587ed06188fcc0d2569267fd1c0363ab9",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    },\r\n    {\r\n      "path": "phase2/analyze_stage2.py",\r\n      "original_sha256": "7654f10f8058e6a26c875e04937b9030a69e629825bc4191f1e9a13d7bf69b18",\r\n      "revised_sha256": "f6d2de7eb7d0c0372b2fd409af419fdf21d95ef0e89cec6cd57139d2e4f51cb0",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    },\r\n    {\r\n      "path": "phase2/analysis/PHASE2_REPORT.md",\r\n      "original_sha256": "a5878baf4127a4b29ed22ac6a9357f4f4785568fcad8966d70ffa98166e2ad1a",\r\n      "revised_sha256": "47227929a1530c9e2cdd3c05abdab90803b2b90dd6878731afca26389b6b9271",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    },\r\n    {\r\n      "path": "phase2/finalize_local.py",\r\n      "original_sha256": "fdd71134d57fc85f3f8ec1a0c3d5e1875d72c4585b92de0a069a62c5c2406751",\r\n      "revised_sha256": "1da5f833ef359ed3eae8f5a3fc10ed0e34f51188cf1e6df841483901748e609a",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    },\r\n    {\r\n      "path": "phase2/RESEARCH_NOTES.md",\r\n      "original_sha256": "f5f9d3265f5e6f26d0eb86b50baaf222b72b0117913b88dbd628f4771338409e",\r\n      "revised_sha256": "605f50ec58635d37386bb07ddce7ded9a18819187ad5e25cb6fb62877d45eaec",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    },\r\n    {\r\n      "path": "analyze.py",\r\n      "original_sha256": "e61eecccfec53bfb9e9b711d2ac3795cef0ff07b90f73447203ea70360359ecb",\r\n      "revised_sha256": "ddbf917d614f861c3fca01b9e7a8782daec0730472c913a68e3756272b7cb2c3",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    },\r\n    {\r\n      "path": "analysis/PRELIMINARY_REPORT.md",\r\n      "original_sha256": "9560c048dfb7e14f53c92d5cba3faa87b4d7e2a1a4316ed8c30936ec6d5c7e97",\r\n      "revised_sha256": "2951c77124c6615489b37a44fb9b2bc31f69c79bd9bab333be8a219b4620a431",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    },\r\n    {\r\n      "path": "build_notebook.py",\r\n      "original_sha256": "9b3504ed480f1535bb44c141359d896a2520e09194ebd47189f8cb8748252269",\r\n      "revised_sha256": "d2c613230a51cd8ade9efae2ff607ab7ce4db21ca51769d4b2e9a23e26cb6a53",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    },\r\n    {\r\n      "path": "README.md",\r\n      "original_sha256": "07a52c2f96133c589455ff5d7310d9fcfe4ac23cc482e2c04b68b78c12ff2a35",\r\n      "revised_sha256": "f3ec27445888971886cf817eeaeaf88dd501eb3178782b078d281caed6c887c5",\r\n      "change": "Document wording only; experimental methods and measured values unchanged."\r\n    }\r\n  ]\r\n}'}
(WORK/'phase3').mkdir(exist_ok=True)
for name,source in PHASE3_SOURCES.items():
    (WORK/'phase3'/name).write_bytes(source.encode('utf-8'))
os.environ['PYTHONPATH']=os.pathsep.join(str(p) for p in [WORK/'phase3',WORK/'phase2',WORK/'vendor',WORK])


## Drive backup

This workflow uses Drive backup certificates in the final analysis. This grants notebook code access to Drive files. The original research notebook obtained Fazal’s approval for project backups. Review the prompt in a fresh notebook. Completed new model exports contain weights/config, while exact optimizer-resume state stays in the temporary runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive',timeout_ms=180000)
DRIVE_BACKUP=Path('/content/drive/MyDrive/mini-k3/2026-10-18')
os.environ['MINIK3_DRIVE_BACKUP']=str(DRIVE_BACKUP)


## Data and checks

Restore token bytes as well as manifests when resuming. For changed code/settings/environment, choose a fresh output directory rather than resuming an old run.


In [ ]:
run([sys.executable,str(WORK/'experiment.py'),'prepare','--upstream',str(WORK/'vendor'),'--data',str(WORK/'data')])
run([sys.executable,str(WORK/'phase2/prepare_holdout.py')])
run([sys.executable,str(WORK/'phase3/prepare_holdout.py')])
run([sys.executable,str(WORK/'test_experiment.py')])
run([sys.executable,str(WORK/'phase3/test_models.py')])


## Screen, lock, confirm

The winner is selected using seed-42 development results only. Confirmation seeds are 17, 123 and 777; no new heldout or WikiText scores enter selection.


In [ ]:
run([sys.executable,str(WORK/'phase3/runner.py'),'screen'])
run([sys.executable,str(WORK/'phase3/runner.py'),'confirmation'])


In [ ]:
run([sys.executable,str(WORK/'phase3/ood_evaluation.py')])
run([sys.executable,str(WORK/'phase3/repeat_timing.py')])
run([sys.executable,str(WORK/'phase3/analyze.py')])


In [ ]:
from IPython.display import display,Markdown,Image
display(Markdown((WORK/'phase3/analysis/PHASE3_REPORT.md').read_text()))
display(Image(filename=str(WORK/'phase3/analysis/phase3_results.png')))
if os.environ.get('MINIK3_DRIVE_BACKUP'):
    import sys
    sys.path.insert(0,str(WORK/'phase2'))
    from backup import backup_tree
    backup_tree(WORK/'phase3',Path(os.environ['MINIK3_DRIVE_BACKUP'])/'phase3',include_checkpoints=False)
